In [1]:
# هذه الخلية لتثبيت المكتبات المطلوبة للمشروع
# لا نستخدم -q حتى يظهر تقدم التحميل أمامنا

!pip install ir_datasets scikit-learn pandas numpy tqdm nltk spacy gensim --progress-bar on

# تحميل موديل spaCy المستخدم في Lemmatization

!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
     - ------------------------------------- 0.5/12.8 MB 575.8 kB/s eta 0:00:22
     - ------------------------------------- 0.5/12.8 MB 575.8 kB/s eta 0:00:22
     - ------------------------------------- 0.5/12.8 MB 575.8 kB/s eta 0:00:22
     -- ------------------------------------ 0.8/12.8 MB 482.5 kB/s eta 0:00:25
     -- ------------------------------------ 0.8/12.8 MB 482.5 kB/s eta 0:00:25
     --- ----------------------------------- 1.0/12.8 MB 566.8 kB/s eta 0:00:21
     --- ----------------------------------- 1.0/12.8 MB 566.8 kB/s eta 0:00:21
     ---

In [2]:
# استيراد المكتبات الأساسية التي سنستخدمها في المشروع

import re
import nltk
import spacy
import pandas as pd
import numpy as np
import ir_datasets
import pickle

from tqdm import tqdm

# مكتبات المعالجة اللغوية

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# مكتبات TF-IDF وقياس التشابه

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# مكتبة Word2Vec من أجل Embedding Representation

from gensim.models import Word2Vec

In [3]:
# تحميل قائمة stop words
# stop words هي كلمات شائعة مثل the و is و are
# غالباً لا تفيد كثيراً في البحث لذلك نحذفها

nltk.download("stopwords")

stop_words = set(stopwords.words("english"))

# تجهيز أداة Stemming

stemmer = PorterStemmer()

# تجهيز spaCy من أجل Lemmatization
# قمنا بتعطيل parser و ner لتسريع التنفيذ

nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

print("Preprocessing tools are ready.")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Preprocessing tools are ready.


In [4]:
# هنا نحدد أسماء مجموعتي البيانات
# لا نستخدم antique لأنها ممنوعة
# نستخدم مجموعتين من ir-datasets تحققان الشرط المطلوب

DATASET_1_NAME = "beir/webis-touche2020/v2"
DATASET_2_NAME = "beir/quora/test"

dataset1 = ir_datasets.load(DATASET_1_NAME)
dataset2 = ir_datasets.load(DATASET_2_NAME)

# التحقق من عدد الوثائق والاستعلامات و qrels في Dataset 1

print("Dataset 1:", DATASET_1_NAME)
print("Documents:", dataset1.docs_count())
print("Queries:", dataset1.queries_count())
print("Qrels:", dataset1.qrels_count())

# التحقق من عدد الوثائق والاستعلامات و qrels في Dataset 2

print("\nDataset 2:", DATASET_2_NAME)
print("Documents:", dataset2.docs_count())
print("Queries:", dataset2.queries_count())
print("Qrels:", dataset2.qrels_count())

Dataset 1: beir/webis-touche2020/v2
Documents: 382545
Queries: 49
Qrels: 2214

Dataset 2: beir/quora/test
Documents: 522931
Queries: 10000
Qrels: 15675


In [5]:
# هذه الدالة تطبق Normalization على النص
# أي تنظيف النص وتوحيد شكله قبل الفهرسة والبحث

def normalize_text(text):
    # تحويل النص إلى string ثم إلى أحرف صغيرة

    text = str(text).lower()
    
    # حذف الروابط

    text = re.sub(r"http\S+|www\S+", " ", text)
    
    # حذف الأرقام والرموز وعلامات الترقيم
    # نبقي فقط الأحرف الإنكليزية والمسافات

    text = re.sub(r"[^a-z\s]", " ", text)
    
    # حذف الفراغات الزائدة

    text = re.sub(r"\s+", " ", text).strip()
    
    return text

In [6]:
# هذه الدالة تطبق خطوات المعالجة التالية:
# 1. Normalization
# 2. Tokenization
# 3. Stop Words Removal
# 4. Stemming

def preprocess_stemming(text):
    # تطبيق Normalization

    text = normalize_text(text)
    
    # Tokenization
    # تقسيم النص إلى كلمات

    words = text.split()
    
    tokens = []
    
    for word in words:
        # حذف stop words والكلمات القصيرة جداً

        if word not in stop_words and len(word) > 2:
            # تطبيق Stemming على الكلمة

            stemmed_word = stemmer.stem(word)
            tokens.append(stemmed_word)
    
    # إعادة الكلمات المعالجة كنص واحد

    return " ".join(tokens)

In [7]:
# هذه الدالة تطبق Lemmatization
# Lemmatization تعيد الكلمة إلى أصلها اللغوي الصحيح

def preprocess_lemmatization(text):
    # تطبيق Normalization

    text = normalize_text(text)
    
    # تمرير النص إلى spaCy

    doc = nlp(text)
    
    tokens = []
    
    for token in doc:
        # أخذ الأصل اللغوي للكلمة

        lemma = token.lemma_.strip()
        
        # حذف stop words والكلمات القصيرة

        if lemma and lemma not in stop_words and len(lemma) > 2:
            tokens.append(lemma)
    
    return " ".join(tokens)

In [9]:
# هذه الخلية لإثبات أن خطوات المعالجة تعمل بشكل صحيح

sample_text = "I am running and studying Information Retrieval Systems!"

print("Original Text:")
print(sample_text)

print("\nAfter Normalization:")
print(normalize_text(sample_text))

print("\nAfter Stemming Preprocessing:")
print(preprocess_stemming(sample_text))

print("\nAfter Lemmatization Preprocessing:")
print(preprocess_lemmatization(sample_text))

Original Text:
I am running and studying Information Retrieval Systems!

After Normalization:
i am running and studying information retrieval systems

After Stemming Preprocessing:
run studi inform retriev system

After Lemmatization Preprocessing:
run study information retrieval system


In [12]:
# تعريف دالة قراءة نص الوثيقة
# بعض الداتاسيت يكون فيها title وبعضها text
# لذلك نجمع الاثنين إذا كانا موجودين

def get_doc_text(doc):
    title = getattr(doc, "title", "")
    text = getattr(doc, "text", "")
    
    return (str(title) + " " + str(text)).strip()

In [13]:
# تعريف دالة تحميل الوثائق من الداتاسيت
# هذه الدالة تستخدم get_doc_text لذلك يجب أن تكون get_doc_text معرفة قبلها

def load_documents(dataset_name, max_docs=3000):
    dataset = ir_datasets.load(dataset_name)
    
    docs = []
    
    for doc in tqdm(dataset.docs_iter(), desc=f"Loading docs from {dataset_name}"):
        docs.append({
            "doc_id": doc.doc_id,
            "text": get_doc_text(doc)
        })
        
        if len(docs) >= max_docs:
            break
    
    return pd.DataFrame(docs)

print("get_doc_text and load_documents are ready.")

get_doc_text and load_documents are ready.


In [14]:
# تحميل عينة صغيرة من كل Dataset
# نبدأ بـ 3000 وثيقة حتى يكون الحجم والتنفيذ مقبولاً

MAX_DOCS = 3000

docs1_df = load_documents(DATASET_1_NAME, MAX_DOCS)
docs2_df = load_documents(DATASET_2_NAME, MAX_DOCS)

print("Dataset 1 loaded sample:", docs1_df.shape)
print("Dataset 2 loaded sample:", docs2_df.shape)

display(docs1_df.head())
display(docs2_df.head())

Loading docs from beir/webis-touche2020/v2: 2999it [00:00, 67309.42it/s]
Loading docs from beir/quora/test: 2999it [00:00, 151567.25it/s]

Dataset 1 loaded sample: (3000, 2)
Dataset 2 loaded sample: (3000, 2)


,doc_id,text
0,c67482ba-2019-04-18T13:32:05Z-00000-000,Contraceptive Forms for High School Students M...
1,c67482ba-2019-04-18T13:32:05Z-00001-000,Contraceptive Forms for High School Students H...
2,c67482ba-2019-04-18T13:32:05Z-00002-000,Contraceptive Forms for High School Students S...
3,c67482ba-2019-04-18T13:32:05Z-00003-000,Contraceptive Forms for High School Students A...
4,4d3d4471-2019-04-18T11:45:01Z-00000-000,Australia should be a more significant country...


,doc_id,text
0,1,What is the step by step guide to invest in sh...
1,2,What is the step by step guide to invest in sh...
2,3,What is the story of Kohinoor (Koh-i-Noor) Dia...
3,4,What would happen if the Indian government sto...
4,5,How can I increase the speed of my internet co...


In [15]:
# Apply preprocessing
# تطبيق المعالجة المسبقة

# نطبق preprocessing على النصوص
# الناتج سنخزنه في عمود processed_text

docs1_df["processed_text"] = [
    preprocess_stemming(text)
    for text in tqdm(docs1_df["text"], desc="Preprocessing Dataset 1")
]

docs2_df["processed_text"] = [
    preprocess_stemming(text)
    for text in tqdm(docs2_df["text"], desc="Preprocessing Dataset 2")
]

print("Dataset 1 after preprocessing:")
display(docs1_df[["doc_id", "text", "processed_text"]].head())

print("Dataset 2 after preprocessing:")
display(docs2_df[["doc_id", "text", "processed_text"]].head())

Preprocessing Dataset 2: 100%|██████████| 3000/3000 [00:00<00:00, 12723.44it/s]

Dataset 1 after preprocessing:


,doc_id,text,processed_text
0,c67482ba-2019-04-18T13:32:05Z-00000-000,Contraceptive Forms for High School Students M...,contracept form high school student oppon forf...
1,c67482ba-2019-04-18T13:32:05Z-00001-000,Contraceptive Forms for High School Students H...,contracept form high school student propos sch...
2,c67482ba-2019-04-18T13:32:05Z-00002-000,Contraceptive Forms for High School Students S...,contracept form high school student school com...
3,c67482ba-2019-04-18T13:32:05Z-00003-000,Contraceptive Forms for High School Students A...,contracept form high school student senior sch...
4,4d3d4471-2019-04-18T11:45:01Z-00000-000,Australia should be a more significant country...,australia signific countri resolut use pro ass...


Dataset 2 after preprocessing:


,doc_id,text,processed_text
0,1,What is the step by step guide to invest in sh...,step step guid invest share market india
1,2,What is the step by step guide to invest in sh...,step step guid invest share market
2,3,What is the story of Kohinoor (Koh-i-Noor) Dia...,stori kohinoor koh noor diamond
3,4,What would happen if the Indian government sto...,would happen indian govern stole kohinoor koh ...
4,5,How can I increase the speed of my internet co...,increas speed internet connect use vpn


In [16]:
# Working samples
# إنشاء عينات العمل

# هذه الجداول سنستخدمها في TF-IDF و Word2Vec و BM25

N_REPRESENTATION_DOCS = 3000

work1_df = docs1_df.head(N_REPRESENTATION_DOCS).copy()
work2_df = docs2_df.head(N_REPRESENTATION_DOCS).copy()

print("work1_df shape:", work1_df.shape)
print("work2_df shape:", work2_df.shape)

print("\nColumns in work1_df:")
print(work1_df.columns)

print("\nColumns in work2_df:")
print(work2_df.columns)

work1_df shape: (3000, 3)
work2_df shape: (3000, 3)

Columns in work1_df:
Index(['doc_id', 'text', 'processed_text'], dtype='object')

Columns in work2_df:
Index(['doc_id', 'text', 'processed_text'], dtype='object')


In [17]:
# VSM TF-IDF representation
# تمثيل الوثائق باستخدام TF-IDF

# كل وثيقة تتحول إلى vector رقمي
# الصفوف تمثل الوثائق
# الأعمدة تمثل الكلمات أو features

tfidf_vectorizer_1 = TfidfVectorizer(max_features=30000)
tfidf_matrix_1 = tfidf_vectorizer_1.fit_transform(work1_df["processed_text"])

tfidf_vectorizer_2 = TfidfVectorizer(max_features=30000)
tfidf_matrix_2 = tfidf_vectorizer_2.fit_transform(work2_df["processed_text"])

print("Dataset 1 TF-IDF matrix shape:", tfidf_matrix_1.shape)
print("Dataset 2 TF-IDF matrix shape:", tfidf_matrix_2.shape)

Dataset 1 TF-IDF matrix shape: (3000, 17463)
Dataset 2 TF-IDF matrix shape: (3000, 3772)


In [18]:
# TF-IDF search
# دالة البحث باستخدام TF-IDF

# نعالج query بنفس طريقة الوثائق
# ثم نحول query إلى TF-IDF vector
# ثم نحسب cosine similarity مع الوثائق

def search_tfidf(query, docs_df, vectorizer, tfidf_matrix, top_k=10):
    processed_query = preprocess_stemming(query)
    
    query_vector = vectorizer.transform([processed_query])
    
    scores = cosine_similarity(query_vector, tfidf_matrix).flatten()
    
    top_indices = scores.argsort()[::-1][:top_k]
    
    results = docs_df.iloc[top_indices][["doc_id", "text"]].copy()
    results["tfidf_score"] = scores[top_indices]
    
    return results

In [19]:
# Test TF-IDF
# تجربة البحث باستخدام TF-IDF

test_query = "what is the best way to lose weight"

print("TF-IDF Results on Dataset 1:")
tfidf_results_1 = search_tfidf(
    query=test_query,
    docs_df=work1_df,
    vectorizer=tfidf_vectorizer_1,
    tfidf_matrix=tfidf_matrix_1,
    top_k=5
)
display(tfidf_results_1)

print("TF-IDF Results on Dataset 2:")
tfidf_results_2 = search_tfidf(
    query=test_query,
    docs_df=work2_df,
    vectorizer=tfidf_vectorizer_2,
    tfidf_matrix=tfidf_matrix_2,
    top_k=5
)
display(tfidf_results_2)

TF-IDF Results on Dataset 1:


,doc_id,text,tfidf_score
1338,a88e4c42-2019-04-18T19:40:07Z-00005-000,I (im_always_right) will lose this debate if I...,0.374399
1336,a88e4c42-2019-04-18T19:40:07Z-00003-000,I (im_always_right) will lose this debate Oh I...,0.359070
1337,a88e4c42-2019-04-18T19:40:07Z-00004-000,I (im_always_right) will lose this debate I ne...,0.283284
30,d6517702-2019-04-18T12:36:24Z-00001-000,Science is the best! Science is the best!,0.262720
1334,a88e4c42-2019-04-18T19:40:07Z-00001-000,I (im_always_right) will lose this debate This...,0.188869


TF-IDF Results on Dataset 2:


,doc_id,text,tfidf_score
2478,2559,What are the best ways to lose weight?,1.000000
2623,2711,What is the best method of losing weight?,0.715545
2477,2558,Why do I not lose weight when I throw up?,0.602135
2624,2712,What are the best way of loose the weight?,0.584073
1633,1693,How do I lose weight without doing any sport?,0.564573


In [20]:
# TF-IDF features
# عرض الكلمات التي تعلمها TF-IDF

# هذا يثبت أن TF-IDF استخرج vocabulary من النصوص المعالجة

feature_names_1 = tfidf_vectorizer_1.get_feature_names_out()
feature_names_2 = tfidf_vectorizer_2.get_feature_names_out()

print("Number of TF-IDF features in Dataset 1:", len(feature_names_1))
print("First 30 TF-IDF features in Dataset 1:")
print(feature_names_1[:30])

print("\nNumber of TF-IDF features in Dataset 2:", len(feature_names_2))
print("First 30 TF-IDF features in Dataset 2:")
print(feature_names_2[:30])

Number of TF-IDF features in Dataset 1: 17463
First 30 TF-IDF features in Dataset 1:
['aaa' 'aaaaah' 'aac' 'aafp' 'aap' 'aaron' 'aauw' 'aba' 'abandon' 'abash'
 'abat' 'abbrevi' 'abc' 'abdomin' 'abduct' 'abel' 'aberdeen' 'aberfeldi'
 'abet' 'abhorr' 'abi' 'abial' 'abid' 'abil' 'abilti' 'abiogenesi'
 'abiogenisi' 'abiogensi' 'abiot' 'abl']

Number of TF-IDF features in Dataset 2: 3772
First 30 TF-IDF features in Dataset 2:
['aamir' 'aap' 'abc' 'abedin' 'abid' 'abil' 'abl' 'abolish' 'abroad'
 'abscond' 'absent' 'absolut' 'abstract' 'abu' 'abus' 'academ' 'academi'
 'acceler' 'accentur' 'accept' 'access' 'accident' 'accord' 'accordion'
 'accounnt' 'account' 'accur' 'accura' 'accus' 'acer']


In [21]:
# Word2Vec tokens
# تجهيز النصوص من أجل Word2Vec

# Word2Vec يحتاج الوثائق كقوائم كلمات
# لذلك نستخدم processed_text ونقسمه إلى tokens

N_WORD2VEC_DOCS = 3000

w2v1_df = work1_df.head(N_WORD2VEC_DOCS).copy()
w2v2_df = work2_df.head(N_WORD2VEC_DOCS).copy()

tokenized_docs_1 = [
    text.split()
    for text in w2v1_df["processed_text"]
]

tokenized_docs_2 = [
    text.split()
    for text in w2v2_df["processed_text"]
]

print("Dataset 1 tokenized documents:", len(tokenized_docs_1))
print("Dataset 2 tokenized documents:", len(tokenized_docs_2))

print("\nExample tokens from Dataset 1:")
print(tokenized_docs_1[0][:20])

Dataset 1 tokenized documents: 3000
Dataset 2 tokenized documents: 3000

Example tokens from Dataset 1:
['contracept', 'form', 'high', 'school', 'student', 'oppon', 'forfeit', 'everi', 'round', 'none', 'argument', 'answer', 'like', 'idea', 'win', 'default', 'tule', 'good', 'student', 'get']


In [22]:
# Train Word2Vec model
# تدريب Word2Vec على الداتاسيت الأولى

# كل كلمة ستتحول إلى vector حجمه 100
# workers = 1 لتجنب مشاكل Windows

word2vec_model_1 = Word2Vec(
    sentences=tokenized_docs_1,
    vector_size=100,
    window=5,
    min_count=2,
    workers=1,
    sg=1,
    epochs=10
)

print("Word2Vec model for Dataset 1 trained successfully.")
print("Vocabulary size Dataset 1:", len(word2vec_model_1.wv))

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Word2Vec model for Dataset 1 trained successfully.
Vocabulary size Dataset 1: 10599


In [23]:
# Train Word2Vec model
# تدريب Word2Vec على الداتاسيت الثانية

# نستخدم نموذج مستقل لأن مفردات كل Dataset مختلفة

word2vec_model_2 = Word2Vec(
    sentences=tokenized_docs_2,
    vector_size=100,
    window=5,
    min_count=2,
    workers=1,
    sg=1,
    epochs=10
)

print("Word2Vec model for Dataset 2 trained successfully.")
print("Vocabulary size Dataset 2:", len(word2vec_model_2.wv))

Word2Vec model for Dataset 2 trained successfully.
Vocabulary size Dataset 2: 2242


In [24]:
# Document vector
# تحويل الوثيقة إلى vector

# Word2Vec يمثل الكلمات وليس الوثائق مباشرة
# لذلك نمثل الوثيقة بمتوسط vectors الكلمات الموجودة فيها

def document_to_vector(tokens, model, vector_size=100):
    vectors = []
    
    for token in tokens:
        if token in model.wv:
            vectors.append(model.wv[token])
    
    if len(vectors) == 0:
        return np.zeros(vector_size)
    
    return np.mean(vectors, axis=0)

In [25]:
# Word2Vec document matrices
# بناء مصفوفات تمثيل الوثائق باستخدام Word2Vec

# كل صف يمثل وثيقة
# كل وثيقة تصبح vector حجمه 100

word2vec_matrix_1 = np.array([
    document_to_vector(tokens, word2vec_model_1, vector_size=100)
    for tokens in tokenized_docs_1
])

word2vec_matrix_2 = np.array([
    document_to_vector(tokens, word2vec_model_2, vector_size=100)
    for tokens in tokenized_docs_2
])

print("Dataset 1 Word2Vec embedding matrix shape:", word2vec_matrix_1.shape)
print("Dataset 2 Word2Vec embedding matrix shape:", word2vec_matrix_2.shape)

Dataset 1 Word2Vec embedding matrix shape: (3000, 100)
Dataset 2 Word2Vec embedding matrix shape: (3000, 100)


In [26]:
# Word2Vec search
# دالة البحث باستخدام Word2Vec

# نحول query إلى vector
# ثم نقارنه مع document vectors باستخدام cosine similarity

def search_word2vec(query, docs_df, word2vec_model, word2vec_matrix, top_k=10):
    processed_query = preprocess_stemming(query)
    
    query_tokens = processed_query.split()
    
    query_vector = document_to_vector(
        query_tokens,
        word2vec_model,
        vector_size=word2vec_matrix.shape[1]
    ).reshape(1, -1)
    
    scores = cosine_similarity(query_vector, word2vec_matrix).flatten()
    
    top_indices = scores.argsort()[::-1][:top_k]
    
    results = docs_df.iloc[top_indices][["doc_id", "text"]].copy()
    results["word2vec_score"] = scores[top_indices]
    
    return results

In [27]:
# Test Word2Vec
# تجربة البحث باستخدام Word2Vec

test_query = "what is the best way to lose weight"

print("Word2Vec Results on Dataset 1:")
word2vec_results_1 = search_word2vec(
    query=test_query,
    docs_df=w2v1_df,
    word2vec_model=word2vec_model_1,
    word2vec_matrix=word2vec_matrix_1,
    top_k=5
)
display(word2vec_results_1)

print("Word2Vec Results on Dataset 2:")
word2vec_results_2 = search_word2vec(
    query=test_query,
    docs_df=w2v2_df,
    word2vec_model=word2vec_model_2,
    word2vec_matrix=word2vec_matrix_2,
    top_k=5
)
display(word2vec_results_2)

Word2Vec Results on Dataset 1:


,doc_id,text,word2vec_score
148,197ac9b-2019-04-18T19:47:57Z-00001-000,Baseball is a better sport than basketball. Ag...,0.824176
1336,a88e4c42-2019-04-18T19:40:07Z-00003-000,I (im_always_right) will lose this debate Oh I...,0.824026
65,a60d2aa5-2019-04-18T17:14:53Z-00002-000,Russell Hantz Should Have Won Survivor: Samoa ...,0.823416
147,197ac9b-2019-04-18T19:47:57Z-00000-000,Baseball is a better sport than basketball. Hi...,0.808622
64,a60d2aa5-2019-04-18T17:14:53Z-00001-000,Russell Hantz Should Have Won Survivor: Samoa ...,0.807318


Word2Vec Results on Dataset 2:


,doc_id,text,word2vec_score
2478,2559,What are the best ways to lose weight?,1.000000
2624,2712,What are the best way of loose the weight?,0.999821
2623,2711,What is the best method of losing weight?,0.999735
1853,1919,How could I gain weight in a healthy way?,0.999579
2360,2441,What is the best way to say 1?,0.999565


In [28]:
# Example word vector
# عرض مثال على vector لكلمة واحدة

# هذا يثبت أن Word2Vec تعلم تمثيلات رقمية للكلمات

example_word = list(word2vec_model_1.wv.index_to_key)[0]

print("Example word from Dataset 1 vocabulary:", example_word)
print("Vector shape for this word:", word2vec_model_1.wv[example_word].shape)
print("First 10 values of the word vector:")
print(word2vec_model_1.wv[example_word][:10])

Example word from Dataset 1 vocabulary: would
Vector shape for this word: (100,)
First 10 values of the word vector:
[ 0.11532664 -0.00365683  0.0889326   0.09118047  0.06400188 -0.20496291
  0.10068333  0.4635396  -0.20731638 -0.13019492]


In [29]:
# BM25 library
# تثبيت واستيراد مكتبة BM25

!pip install rank-bm25 --progress-bar on

from rank_bm25 import BM25Okapi

In [30]:
# BM25 tokenized corpus
# تجهيز الوثائق من أجل BM25

# BM25 يحتاج كل وثيقة على شكل قائمة كلمات
# لذلك نستخدم processed_text ونقسمه إلى tokens

tokenized_corpus_1 = [
    text.split()
    for text in work1_df["processed_text"]
]

tokenized_corpus_2 = [
    text.split()
    for text in work2_df["processed_text"]
]

print("Dataset 1 tokenized documents:", len(tokenized_corpus_1))
print("Dataset 2 tokenized documents:", len(tokenized_corpus_2))

print("\nExample tokenized document from Dataset 1:")
print(tokenized_corpus_1[0][:20])

Dataset 1 tokenized documents: 3000
Dataset 2 tokenized documents: 3000

Example tokenized document from Dataset 1:
['contracept', 'form', 'high', 'school', 'student', 'oppon', 'forfeit', 'everi', 'round', 'none', 'argument', 'answer', 'like', 'idea', 'win', 'default', 'tule', 'good', 'student', 'get']


In [31]:
# Build BM25 models
# بناء نموذج BM25 لكل Dataset

# k1 يتحكم بتأثير تكرار الكلمة داخل الوثيقة
# b يتحكم بتأثير طول الوثيقة

bm25_model_1 = BM25Okapi(
    tokenized_corpus_1,
    k1=1.5,
    b=0.75
)

bm25_model_2 = BM25Okapi(
    tokenized_corpus_2,
    k1=1.5,
    b=0.75
)

print("BM25 model for Dataset 1 is ready.")
print("BM25 model for Dataset 2 is ready.")

BM25 model for Dataset 1 is ready.
BM25 model for Dataset 2 is ready.


In [32]:
# BM25 search
# دالة البحث باستخدام BM25

# الدالة تسمح بتغيير k1 و b
# وهذا مهم لتغطية ملاحظة المشروع

def search_bm25(query, docs_df, tokenized_corpus, top_k=10, k1=1.5, b=0.75):
    processed_query = preprocess_stemming(query)
    
    tokenized_query = processed_query.split()
    
    bm25_model = BM25Okapi(
        tokenized_corpus,
        k1=k1,
        b=b
    )
    
    scores = bm25_model.get_scores(tokenized_query)
    scores = np.array(scores)
    
    top_indices = scores.argsort()[::-1][:top_k]
    
    results = docs_df.iloc[top_indices][["doc_id", "text"]].copy()
    results["bm25_score"] = scores[top_indices]
    results["k1"] = k1
    results["b"] = b
    
    return results

In [33]:
# Test BM25
# تجربة البحث باستخدام BM25 على الداتاسيتين

test_query = "what is the best way to lose weight"

print("BM25 Results on Dataset 1:")
bm25_results_1 = search_bm25(
    query=test_query,
    docs_df=work1_df,
    tokenized_corpus=tokenized_corpus_1,
    top_k=5,
    k1=1.5,
    b=0.75
)
display(bm25_results_1)

print("BM25 Results on Dataset 2:")
bm25_results_2 = search_bm25(
    query=test_query,
    docs_df=work2_df,
    tokenized_corpus=tokenized_corpus_2,
    top_k=5,
    k1=1.5,
    b=0.75
)
display(bm25_results_2)

BM25 Results on Dataset 1:


,doc_id,text,bm25_score,k1,b
1336,a88e4c42-2019-04-18T19:40:07Z-00003-000,I (im_always_right) will lose this debate Oh I...,8.370463,1.5,0.75
1061,2c5282d9-2019-04-18T19:39:34Z-00001-000,Does he fit the definition of a terorist? Note...,7.876333,1.5,0.75
512,f2f66016-2019-04-18T16:48:35Z-00002-000,Pokemon is freakin creepy. You disregard the k...,7.796105,1.5,0.75
1047,8840debf-2019-04-18T13:05:37Z-00004-000,swimming is a better sport than soccer So my o...,7.373301,1.5,0.75
65,a60d2aa5-2019-04-18T17:14:53Z-00002-000,Russell Hantz Should Have Won Survivor: Samoa ...,7.096017,1.5,0.75


BM25 Results on Dataset 2:


,doc_id,text,bm25_score,k1,b
2478,2559,What are the best ways to lose weight?,18.560818,1.5,0.75
2623,2711,What is the best method of losing weight?,14.414871,1.5,0.75
2477,2558,Why do I not lose weight when I throw up?,12.947588,1.5,0.75
2624,2712,What are the best way of loose the weight?,12.566060,1.5,0.75
2225,2303,How do I lose weight and gain muscle?,11.713884,1.5,0.75


In [34]:
# Processed query
# عرض query قبل وبعد المعالجة

# هذا يثبت أننا نستخدم نفس preprocessing مع query والوثائق

print("Original Query:")
print(test_query)

print("\nProcessed Query:")
print(preprocess_stemming(test_query))

Original Query:
what is the best way to lose weight

Processed Query:
best way lose weight


In [35]:
# Change BM25 parameters
# تغيير معاملات BM25

# نجرب نفس query بقيم مختلفة لإثبات أن k1 و b قابلين للتعديل

bm25_results_changed_params = search_bm25(
    query=test_query,
    docs_df=work1_df,
    tokenized_corpus=tokenized_corpus_1,
    top_k=5,
    k1=2.0,
    b=0.50
)

print("BM25 Results on Dataset 1 with changed parameters:")
print("k1 = 2.0")
print("b = 0.50")

display(bm25_results_changed_params)

BM25 Results on Dataset 1 with changed parameters:
k1 = 2.0
b = 0.50


,doc_id,text,bm25_score,k1,b
1336,a88e4c42-2019-04-18T19:40:07Z-00003-000,I (im_always_right) will lose this debate Oh I...,8.547875,2.0,0.5
1061,2c5282d9-2019-04-18T19:39:34Z-00001-000,Does he fit the definition of a terorist? Note...,8.159634,2.0,0.5
1793,be5d28dd-2019-04-18T17:07:11Z-00001-000,Indiscipline does lead to a decline in moral v...,7.836772,2.0,0.5
1335,a88e4c42-2019-04-18T19:40:07Z-00002-000,I (im_always_right) will lose this debate Let'...,7.676489,2.0,0.5
1338,a88e4c42-2019-04-18T19:40:07Z-00005-000,I (im_always_right) will lose this debate if I...,7.658999,2.0,0.5


In [36]:
# Compare BM25 parameters
# مقارنة القيم الافتراضية والقيم المعدلة

# الهدف إثبات أن تغيير المعاملات يؤثر على scoring

print("Default BM25 Parameters:")
print("k1 = 1.5")
print("b = 0.75")

display(bm25_results_1[["doc_id", "bm25_score", "k1", "b", "text"]])

print("\nChanged BM25 Parameters:")
print("k1 = 2.0")
print("b = 0.50")

display(bm25_results_changed_params[["doc_id", "bm25_score", "k1", "b", "text"]])

Default BM25 Parameters:
k1 = 1.5
b = 0.75


,doc_id,bm25_score,k1,b,text
1336,a88e4c42-2019-04-18T19:40:07Z-00003-000,8.370463,1.5,0.75,I (im_always_right) will lose this debate Oh I...
1061,2c5282d9-2019-04-18T19:39:34Z-00001-000,7.876333,1.5,0.75,Does he fit the definition of a terorist? Note...
512,f2f66016-2019-04-18T16:48:35Z-00002-000,7.796105,1.5,0.75,Pokemon is freakin creepy. You disregard the k...
1047,8840debf-2019-04-18T13:05:37Z-00004-000,7.373301,1.5,0.75,swimming is a better sport than soccer So my o...
65,a60d2aa5-2019-04-18T17:14:53Z-00002-000,7.096017,1.5,0.75,Russell Hantz Should Have Won Survivor: Samoa ...



Changed BM25 Parameters:
k1 = 2.0
b = 0.50


,doc_id,bm25_score,k1,b,text
1336,a88e4c42-2019-04-18T19:40:07Z-00003-000,8.547875,2.0,0.5,I (im_always_right) will lose this debate Oh I...
1061,2c5282d9-2019-04-18T19:39:34Z-00001-000,8.159634,2.0,0.5,Does he fit the definition of a terorist? Note...
1793,be5d28dd-2019-04-18T17:07:11Z-00001-000,7.836772,2.0,0.5,Indiscipline does lead to a decline in moral v...
1335,a88e4c42-2019-04-18T19:40:07Z-00002-000,7.676489,2.0,0.5,I (im_always_right) will lose this debate Let'...
1338,a88e4c42-2019-04-18T19:40:07Z-00005-000,7.658999,2.0,0.5,I (im_always_right) will lose this debate if I...


In [37]:
# Save results
# حفظ النتائج والملفات المهمة

# هذه الخلية تساعدك حتى لا تعيد كل شيء من الصفر لاحقاً

docs1_df.to_csv("dataset1_processed_docs.csv", index=False)
docs2_df.to_csv("dataset2_processed_docs.csv", index=False)

work1_df.to_csv("work_dataset1.csv", index=False)
work2_df.to_csv("work_dataset2.csv", index=False)

tfidf_results_1.to_csv("tfidf_results_dataset1.csv", index=False)
tfidf_results_2.to_csv("tfidf_results_dataset2.csv", index=False)

word2vec_results_1.to_csv("word2vec_results_dataset1.csv", index=False)
word2vec_results_2.to_csv("word2vec_results_dataset2.csv", index=False)

bm25_results_1.to_csv("bm25_results_dataset1.csv", index=False)
bm25_results_2.to_csv("bm25_results_dataset2.csv", index=False)

word2vec_model_1.save("word2vec_model_dataset1.model")
word2vec_model_2.save("word2vec_model_dataset2.model")

np.save("word2vec_matrix_1.npy", word2vec_matrix_1)
np.save("word2vec_matrix_2.npy", word2vec_matrix_2)

print("Important files saved successfully.")

Important files saved successfully.


In [40]:
# Hybrid Imports
# استيراد المكتبات اللازمة للتمثيل الهجين

# نحتاج numpy للحسابات والترتيب
# ونحتاج cosine_similarity لحساب التشابه بين query والوثائق

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi

In [41]:
# Score Normalization
# تطبيع الدرجات

# في Parallel Hybrid سنجمع درجات BM25 و Word2Vec
# لكن درجات BM25 و Word2Vec ليست على نفس المقياس
# لذلك نستخدم Min-Max Normalization لتحويل القيم إلى مجال بين 0 و 1

def min_max_normalize(scores):
    scores = np.array(scores)
    
    min_score = scores.min()
    max_score = scores.max()
    
    if max_score - min_score == 0:
        return np.zeros_like(scores)
    
    return (scores - min_score) / (max_score - min_score)

In [42]:
# Serial Hybrid Search
# البحث الهجين التسلسلي

# الفكرة:
# أولاً نستخدم BM25 لجلب عدد من الوثائق المرشحة
# ثم نستخدم Word2Vec لإعادة ترتيب هذه الوثائق المرشحة فقط

def serial_hybrid_search(
    query,
    docs_df,
    tokenized_corpus,
    word2vec_model,
    word2vec_matrix,
    top_k=10,
    candidate_k=100,
    k1=1.5,
    b=0.75
):
    # معالجة query بنفس طريقة معالجة الوثائق

    processed_query = preprocess_stemming(query)
    tokenized_query = processed_query.split()
    
    # المرحلة الأولى:
    # استخدام BM25 لجلب أفضل candidate_k وثيقة

    bm25_model = BM25Okapi(
        tokenized_corpus,
        k1=k1,
        b=b
    )
    
    bm25_scores = np.array(
        bm25_model.get_scores(tokenized_query)
    )
    
    candidate_k = min(candidate_k, len(docs_df))
    
    candidate_indices = bm25_scores.argsort()[::-1][:candidate_k]
    
    # المرحلة الثانية:
    # تحويل query إلى Word2Vec vector

    query_tokens = processed_query.split()
    
    query_vector = document_to_vector(
        query_tokens,
        word2vec_model,
        vector_size=word2vec_matrix.shape[1]
    ).reshape(1, -1)
    
    # نأخذ embeddings للوثائق المرشحة فقط

    candidate_embeddings = word2vec_matrix[candidate_indices]
    
    # حساب التشابه الدلالي بين query والوثائق المرشحة

    embedding_scores = cosine_similarity(
        query_vector,
        candidate_embeddings
    ).flatten()
    
    # إعادة ترتيب المرشحين حسب Word2Vec score

    reranked_positions = embedding_scores.argsort()[::-1][:top_k]
    
    final_indices = candidate_indices[reranked_positions]
    
    results = docs_df.iloc[final_indices][["doc_id", "text"]].copy()
    
    results["bm25_candidate_score"] = bm25_scores[final_indices]
    results["word2vec_rerank_score"] = embedding_scores[reranked_positions]
    results["candidate_k"] = candidate_k
    results["k1"] = k1
    results["b"] = b
    
    return results

In [43]:
# Test Serial Hybrid on Dataset 1
# تجربة البحث الهجين التسلسلي على الداتاسيت الأولى

test_query = "what is the best way to lose weight"

serial_results_1 = serial_hybrid_search(
    query=test_query,
    docs_df=work1_df,
    tokenized_corpus=tokenized_corpus_1,
    word2vec_model=word2vec_model_1,
    word2vec_matrix=word2vec_matrix_1,
    top_k=5,
    candidate_k=100,
    k1=1.5,
    b=0.75
)

print("Serial Hybrid Results on Dataset 1:")
display(serial_results_1)

Serial Hybrid Results on Dataset 1:


,doc_id,text,bm25_candidate_score,word2vec_rerank_score,candidate_k,k1,b
1336,a88e4c42-2019-04-18T19:40:07Z-00003-000,I (im_always_right) will lose this debate Oh I...,8.370463,0.824026,100,1.5,0.75
65,a60d2aa5-2019-04-18T17:14:53Z-00002-000,Russell Hantz Should Have Won Survivor: Samoa ...,7.096017,0.823416,100,1.5,0.75
64,a60d2aa5-2019-04-18T17:14:53Z-00001-000,Russell Hantz Should Have Won Survivor: Samoa ...,4.098500,0.807318,100,1.5,0.75
2291,a8be11b0-2019-04-18T18:58:55Z-00000-000,Cannabis should be legalised Tiresome. I was h...,4.802697,0.805814,100,1.5,0.75
149,197ac9b-2019-04-18T19:47:57Z-00002-000,Baseball is a better sport than basketball. Al...,4.237482,0.803353,100,1.5,0.75


In [44]:
# Test Serial Hybrid on Dataset 2
# تجربة البحث الهجين التسلسلي على الداتاسيت الثانية

serial_results_2 = serial_hybrid_search(
    query=test_query,
    docs_df=work2_df,
    tokenized_corpus=tokenized_corpus_2,
    word2vec_model=word2vec_model_2,
    word2vec_matrix=word2vec_matrix_2,
    top_k=5,
    candidate_k=100,
    k1=1.5,
    b=0.75
)

print("Serial Hybrid Results on Dataset 2:")
display(serial_results_2)

Serial Hybrid Results on Dataset 2:


,doc_id,text,bm25_candidate_score,word2vec_rerank_score,candidate_k,k1,b
2478,2559,What are the best ways to lose weight?,18.560818,1.000000,100,1.5,0.75
2624,2712,What are the best way of loose the weight?,12.566060,0.999821,100,1.5,0.75
2623,2711,What is the best method of losing weight?,14.414871,0.999735,100,1.5,0.75
1853,1919,How could I gain weight in a healthy way?,9.006859,0.999579,100,1.5,0.75
2360,2441,What is the best way to say 1?,7.568052,0.999565,100,1.5,0.75


In [45]:
# Parallel Hybrid Search
# البحث الهجين المتوازي

# الفكرة:
# نحسب BM25 score لكل الوثائق
# ونحسب Word2Vec score لكل الوثائق
# ثم ندمج الدرجات باستخدام Fusion Method

# Fusion Method المستخدم:
# Weighted Sum Fusion

# final_score = alpha * BM25_score + (1 - alpha) * Word2Vec_score

def parallel_hybrid_search(
    query,
    docs_df,
    tokenized_corpus,
    word2vec_model,
    word2vec_matrix,
    top_k=10,
    alpha=0.6,
    k1=1.5,
    b=0.75
):
    # معالجة query

    processed_query = preprocess_stemming(query)
    tokenized_query = processed_query.split()
    
    # حساب BM25 scores

    bm25_model = BM25Okapi(
        tokenized_corpus,
        k1=k1,
        b=b
    )
    
    bm25_scores = np.array(
        bm25_model.get_scores(tokenized_query)
    )
    
    # حساب Word2Vec scores

    query_tokens = processed_query.split()
    
    query_vector = document_to_vector(
        query_tokens,
        word2vec_model,
        vector_size=word2vec_matrix.shape[1]
    ).reshape(1, -1)
    
    word2vec_scores = cosine_similarity(
        query_vector,
        word2vec_matrix
    ).flatten()
    
    # تطبيع الدرجات قبل الدمج
    # لأن BM25 و Word2Vec لهما مقاييس مختلفة

    bm25_norm = min_max_normalize(bm25_scores)
    word2vec_norm = min_max_normalize(word2vec_scores)
    
    # Fusion Method
    # دمج الدرجات باستخدام weighted sum

    final_scores = alpha * bm25_norm + (1 - alpha) * word2vec_norm
    
    # ترتيب الوثائق حسب final_score

    top_indices = final_scores.argsort()[::-1][:top_k]
    
    results = docs_df.iloc[top_indices][["doc_id", "text"]].copy()
    
    results["bm25_score_norm"] = bm25_norm[top_indices]
    results["word2vec_score_norm"] = word2vec_norm[top_indices]
    results["final_hybrid_score"] = final_scores[top_indices]
    results["alpha"] = alpha
    results["k1"] = k1
    results["b"] = b
    
    return results

In [46]:
# Test Parallel Hybrid on Dataset 1
# تجربة البحث الهجين المتوازي على الداتاسيت الأولى

parallel_results_1 = parallel_hybrid_search(
    query=test_query,
    docs_df=work1_df,
    tokenized_corpus=tokenized_corpus_1,
    word2vec_model=word2vec_model_1,
    word2vec_matrix=word2vec_matrix_1,
    top_k=5,
    alpha=0.6,
    k1=1.5,
    b=0.75
)

print("Parallel Hybrid Results on Dataset 1:")
display(parallel_results_1)

Parallel Hybrid Results on Dataset 1:


,doc_id,text,bm25_score_norm,word2vec_score_norm,final_hybrid_score,alpha,k1,b
1336,a88e4c42-2019-04-18T19:40:07Z-00003-000,I (im_always_right) will lose this debate Oh I...,1.000000,0.999739,0.999896,0.6,1.5,0.75
1061,2c5282d9-2019-04-18T19:39:34Z-00001-000,Does he fit the definition of a terorist? Note...,0.940967,0.867580,0.911612,0.6,1.5,0.75
65,a60d2aa5-2019-04-18T17:14:53Z-00002-000,Russell Hantz Should Have Won Survivor: Samoa ...,0.847745,0.998682,0.908120,0.6,1.5,0.75
1047,8840debf-2019-04-18T13:05:37Z-00004-000,swimming is a better sport than soccer So my o...,0.880871,0.943985,0.906117,0.6,1.5,0.75
512,f2f66016-2019-04-18T16:48:35Z-00002-000,Pokemon is freakin creepy. You disregard the k...,0.931383,0.845520,0.897038,0.6,1.5,0.75


In [47]:
# Test Parallel Hybrid on Dataset 2
# تجربة البحث الهجين المتوازي على الداتاسيت الثانية

parallel_results_2 = parallel_hybrid_search(
    query=test_query,
    docs_df=work2_df,
    tokenized_corpus=tokenized_corpus_2,
    word2vec_model=word2vec_model_2,
    word2vec_matrix=word2vec_matrix_2,
    top_k=5,
    alpha=0.6,
    k1=1.5,
    b=0.75
)

print("Parallel Hybrid Results on Dataset 2:")
display(parallel_results_2)

Parallel Hybrid Results on Dataset 2:


,doc_id,text,bm25_score_norm,word2vec_score_norm,final_hybrid_score,alpha,k1,b
2478,2559,What are the best ways to lose weight?,1.000000,1.000000,1.000000,0.6,1.5,0.75
2623,2711,What is the best method of losing weight?,0.776629,0.999752,0.865878,0.6,1.5,0.75
2477,2558,Why do I not lose weight when I throw up?,0.697576,0.999248,0.818245,0.6,1.5,0.75
2624,2712,What are the best way of loose the weight?,0.677021,0.999832,0.806145,0.6,1.5,0.75
1633,1693,How do I lose weight without doing any sport?,0.631108,0.999326,0.778395,0.6,1.5,0.75


In [48]:
# Test different alpha values
# تجربة أكثر من وزن دمج

# هذه الخلية تثبت أن Fusion Method قابل للتعديل
# alpha = 0.8 يعني نعطي BM25 وزناً أكبر
# alpha = 0.3 يعني نعطي Word2Vec وزناً أكبر

print("Parallel Hybrid with alpha = 0.8")
parallel_alpha_08 = parallel_hybrid_search(
    query=test_query,
    docs_df=work1_df,
    tokenized_corpus=tokenized_corpus_1,
    word2vec_model=word2vec_model_1,
    word2vec_matrix=word2vec_matrix_1,
    top_k=5,
    alpha=0.8,
    k1=1.5,
    b=0.75
)

display(parallel_alpha_08)

print("\nParallel Hybrid with alpha = 0.3")
parallel_alpha_03 = parallel_hybrid_search(
    query=test_query,
    docs_df=work1_df,
    tokenized_corpus=tokenized_corpus_1,
    word2vec_model=word2vec_model_1,
    word2vec_matrix=word2vec_matrix_1,
    top_k=5,
    alpha=0.3,
    k1=1.5,
    b=0.75
)

display(parallel_alpha_03)

Parallel Hybrid with alpha = 0.8


,doc_id,text,bm25_score_norm,word2vec_score_norm,final_hybrid_score,alpha,k1,b
1336,a88e4c42-2019-04-18T19:40:07Z-00003-000,I (im_always_right) will lose this debate Oh I...,1.000000,0.999739,0.999948,0.8,1.5,0.75
1061,2c5282d9-2019-04-18T19:39:34Z-00001-000,Does he fit the definition of a terorist? Note...,0.940967,0.867580,0.926290,0.8,1.5,0.75
512,f2f66016-2019-04-18T16:48:35Z-00002-000,Pokemon is freakin creepy. You disregard the k...,0.931383,0.845520,0.914210,0.8,1.5,0.75
1047,8840debf-2019-04-18T13:05:37Z-00004-000,swimming is a better sport than soccer So my o...,0.880871,0.943985,0.893494,0.8,1.5,0.75
65,a60d2aa5-2019-04-18T17:14:53Z-00002-000,Russell Hantz Should Have Won Survivor: Samoa ...,0.847745,0.998682,0.877932,0.8,1.5,0.75



Parallel Hybrid with alpha = 0.3


,doc_id,text,bm25_score_norm,word2vec_score_norm,final_hybrid_score,alpha,k1,b
1336,a88e4c42-2019-04-18T19:40:07Z-00003-000,I (im_always_right) will lose this debate Oh I...,1.000000,0.999739,0.999817,0.3,1.5,0.75
65,a60d2aa5-2019-04-18T17:14:53Z-00002-000,Russell Hantz Should Have Won Survivor: Samoa ...,0.847745,0.998682,0.953401,0.3,1.5,0.75
1047,8840debf-2019-04-18T13:05:37Z-00004-000,swimming is a better sport than soccer So my o...,0.880871,0.943985,0.925051,0.3,1.5,0.75
1338,a88e4c42-2019-04-18T19:40:07Z-00005-000,I (im_always_right) will lose this debate if I...,0.820935,0.962231,0.919842,0.3,1.5,0.75
1831,e737a99c-2019-04-18T12:18:51Z-00000-000,"Rap Battle Okay, yes, I admit that I misspelle...",0.801996,0.949679,0.905374,0.3,1.5,0.75


In [51]:
# Save Hybrid Results
# حفظ نتائج التمثيل الهجين

serial_results_1.to_csv("serial_hybrid_results_dataset1.csv", index=False)
serial_results_2.to_csv("serial_hybrid_results_dataset2.csv", index=False)

parallel_results_1.to_csv("parallel_hybrid_results_dataset1.csv", index=False)
parallel_results_2.to_csv("parallel_hybrid_results_dataset2.csv", index=False)

parallel_alpha_08.to_csv("parallel_hybrid_alpha_08.csv", index=False)
parallel_alpha_03.to_csv("parallel_hybrid_alpha_03.csv", index=False)

print("Hybrid results saved successfully.")

Hybrid results saved successfully.


In [52]:
# Install ipywidgets
# تثبيت مكتبة ipywidgets لإنشاء واجهة تفاعلية داخل Jupyter

!pip install ipywidgets --progress-bar on

In [53]:
# Import widgets
# استيراد أدوات الواجهة التفاعلية

import ipywidgets as widgets
from IPython.display import display, clear_output

In [54]:
# Hybrid Search GUI
# واجهة رسومية بسيطة لاختيار نوع البحث الهجين

# هذه الواجهة تحقق شرط توفير خيار للمستخدم
# المستخدم يختار Dataset
# ويختار Serial Hybrid أو Parallel Hybrid
# ثم يحدد query والمعاملات ويضغط Search

dataset_dropdown = widgets.Dropdown(
    options=[
        ("Dataset 1", "dataset1"),
        ("Dataset 2", "dataset2")
    ],
    value="dataset1",
    description="Dataset:"
)

method_dropdown = widgets.Dropdown(
    options=[
        ("Serial Hybrid", "serial"),
        ("Parallel Hybrid", "parallel")
    ],
    value="serial",
    description="Method:"
)

query_text = widgets.Text(
    value="what is the best way to lose weight",
    placeholder="Enter your query",
    description="Query:",
    layout=widgets.Layout(width="700px")
)

top_k_slider = widgets.IntSlider(
    value=5,
    min=1,
    max=20,
    step=1,
    description="Top K:"
)

candidate_k_slider = widgets.IntSlider(
    value=100,
    min=10,
    max=500,
    step=10,
    description="Candidate K:"
)

k1_slider = widgets.FloatSlider(
    value=1.5,
    min=0.5,
    max=3.0,
    step=0.1,
    description="k1:"
)

b_slider = widgets.FloatSlider(
    value=0.75,
    min=0.0,
    max=1.0,
    step=0.05,
    description="b:"
)

alpha_slider = widgets.FloatSlider(
    value=0.6,
    min=0.0,
    max=1.0,
    step=0.05,
    description="Alpha:"
)

search_button = widgets.Button(
    description="Search",
    button_style="success"
)

output_area = widgets.Output()

In [55]:
# Button action
# تعريف ماذا يحدث عند الضغط على زر Search

def on_search_button_clicked(button):
    with output_area:
        clear_output()
        
        selected_dataset = dataset_dropdown.value
        selected_method = method_dropdown.value
        query = query_text.value
        top_k = top_k_slider.value
        candidate_k = candidate_k_slider.value
        k1 = k1_slider.value
        b = b_slider.value
        alpha = alpha_slider.value
        
        # اختيار الداتاسيت حسب اختيار المستخدم
        
        if selected_dataset == "dataset1":
            docs_df = work1_df
            tokenized_corpus = tokenized_corpus_1
            word2vec_model = word2vec_model_1
            word2vec_matrix = word2vec_matrix_1
        else:
            docs_df = work2_df
            tokenized_corpus = tokenized_corpus_2
            word2vec_model = word2vec_model_2
            word2vec_matrix = word2vec_matrix_2
        
        print("Query:", query)
        print("Selected Dataset:", selected_dataset)
        print("Selected Method:", selected_method)
        print("Top K:", top_k)
        print("k1:", k1)
        print("b:", b)
        
        # إذا اختار المستخدم Serial Hybrid
        
        if selected_method == "serial":
            print("Candidate K:", candidate_k)
            
            results = serial_hybrid_search(
                query=query,
                docs_df=docs_df,
                tokenized_corpus=tokenized_corpus,
                word2vec_model=word2vec_model,
                word2vec_matrix=word2vec_matrix,
                top_k=top_k,
                candidate_k=candidate_k,
                k1=k1,
                b=b
            )
            
            print("\nSerial Hybrid Results:")
            display(results)
        
        # إذا اختار المستخدم Parallel Hybrid
        
        elif selected_method == "parallel":
            print("Alpha:", alpha)
            
            results = parallel_hybrid_search(
                query=query,
                docs_df=docs_df,
                tokenized_corpus=tokenized_corpus,
                word2vec_model=word2vec_model,
                word2vec_matrix=word2vec_matrix,
                top_k=top_k,
                alpha=alpha,
                k1=k1,
                b=b
            )
            
            print("\nParallel Hybrid Results:")
            display(results)


search_button.on_click(on_search_button_clicked)

print("Button action is ready.")

Button action is ready.


In [56]:
# Display GUI
# عرض الواجهة التفاعلية

# هذه هي الواجهة التي يستخدمها المستخدم لاختيار:
# Dataset
# Serial أو Parallel
# query
# top_k
# candidate_k
# k1
# b
# alpha

gui = widgets.VBox([
    widgets.HTML("<h3>Hybrid Search Interface</h3>"),
    dataset_dropdown,
    method_dropdown,
    query_text,
    top_k_slider,
    candidate_k_slider,
    k1_slider,
    b_slider,
    alpha_slider,
    search_button,
    output_area
])

display(gui)

In [60]:
# بناء دالة لإنشاء Inverted Index
# الفهرس المعكوس يربط كل كلمة بقائمة الوثائق التي تحتوي هذه الكلمة
# سنخزن أيضاً عدد تكرار الكلمة داخل كل وثيقة

def build_inverted_index(docs_df):
    inverted_index = {}
    
    for row_index, row in docs_df.iterrows():
        doc_id = row["doc_id"]
        text = row["processed_text"]
        
        tokens = text.split()
        
        for token in tokens:
            if token not in inverted_index:
                inverted_index[token] = {}
            
            if doc_id not in inverted_index[token]:
                inverted_index[token][doc_id] = 0
            
            inverted_index[token][doc_id] += 1
    
    return inverted_index

In [61]:
# بناء Inverted Index لكل Dataset
# نستخدم النص المعالج processed_text لأنه جاهز للفهرسة

inverted_index_1 = build_inverted_index(work1_df)
inverted_index_2 = build_inverted_index(work2_df)

print("Dataset 1 index terms:", len(inverted_index_1))
print("Dataset 2 index terms:", len(inverted_index_2))

Dataset 1 index terms: 17465
Dataset 2 index terms: 3772


In [63]:
# البحث باستخدام Inverted Index
# هذه الدالة تأخذ query
# ثم تعالجها بنفس طريقة معالجة الوثائق
# ثم تبحث عن كلمات query داخل الفهرس
# ثم ترجع الوثائق المرشحة

def search_inverted_index(query, docs_df, inverted_index, top_k=10):
    processed_query = preprocess_stemming(query)
    query_tokens = processed_query.split()
    
    candidate_scores = {}
    
    for token in query_tokens:
        if token in inverted_index:
            posting_list = inverted_index[token]
            
            for doc_id, freq in posting_list.items():
                if doc_id not in candidate_scores:
                    candidate_scores[doc_id] = 0
                
                candidate_scores[doc_id] += freq
    
    sorted_candidates = sorted(
        candidate_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )[:top_k]
    
    result_doc_ids = [doc_id for doc_id, score in sorted_candidates]
    result_scores = {doc_id: score for doc_id, score in sorted_candidates}
    
    results = docs_df[docs_df["doc_id"].isin(result_doc_ids)][["doc_id", "text"]].copy()
    
    results["index_score"] = results["doc_id"].map(result_scores)
    
    results = results.sort_values(by="index_score", ascending=False)
    
    return results

In [64]:
# تجربة البحث باستخدام Inverted Index على Dataset 1

test_query = "what is the best way to lose weight"

index_results_1 = search_inverted_index(
    query=test_query,
    docs_df=work1_df,
    inverted_index=inverted_index_1,
    top_k=5
)

print("Inverted Index Results on Dataset 1:")
display(index_results_1)

Inverted Index Results on Dataset 1:


,doc_id,text,index_score
1337,a88e4c42-2019-04-18T19:40:07Z-00004-000,I (im_always_right) will lose this debate I ne...,12
125,a617b5f8-2019-04-18T12:37:07Z-00002-000,The US Constitution Should Be a Living Documen...,11
612,ef778452-2019-04-18T15:25:55Z-00006-000,Evolution vs creationism I will support creati...,11
409,939a10f1-2019-04-18T17:47:16Z-00001-000,It should not be acceptable for a woman to be ...,10
2606,f7c26c8f-2019-04-18T19:52:26Z-00005-000,You should always be patriotic. [Note: I am no...,10


In [65]:
# تجربة البحث باستخدام Inverted Index على Dataset 2

index_results_2 = search_inverted_index(
    query=test_query,
    docs_df=work2_df,
    inverted_index=inverted_index_2,
    top_k=5
)

print("Inverted Index Results on Dataset 2:")
display(index_results_2)

Inverted Index Results on Dataset 2:


,doc_id,text,index_score
2478,2559,What are the best ways to lose weight?,4
396,406,What is the best way to start robotics? Which ...,3
2623,2711,What is the best method of losing weight?,3
2624,2712,What are the best way of loose the weight?,3
2811,2906,What is the best place to visit in the world? ...,3


In [66]:
# دالة لحساب إحصائيات عن الفهرس
# هذه الإحصائيات تساعدنا على إثبات أن الفهرسة تمت فعلاً

def index_statistics(inverted_index, docs_df):
    number_of_documents = len(docs_df)
    number_of_terms = len(inverted_index)
    
    posting_lengths = [
        len(posting_list)
        for posting_list in inverted_index.values()
    ]
    
    total_postings = sum(posting_lengths)
    average_posting_length = total_postings / number_of_terms if number_of_terms > 0 else 0
    
    return {
        "number_of_documents": number_of_documents,
        "number_of_terms": number_of_terms,
        "total_postings": total_postings,
        "average_posting_length": average_posting_length
    }

In [72]:
# عرض إحصائيات الفهرس لكل Dataset

index_stats_1 = index_statistics(inverted_index_1, work1_df)
index_stats_2 = index_statistics(inverted_index_2, work2_df)

print("Dataset 1 Index Statistics:")
for key, value in index_stats_1.items():
    print(key, ":", value)

print("\nDataset 2 Index Statistics:")
for key, value in index_stats_2.items():
    print(key, ":", value)

Dataset 1 Index Statistics:
number_of_documents : 3000
number_of_terms : 17465
total_postings : 274542
average_posting_length : 15.719553392499284

Dataset 2 Index Statistics:
number_of_documents : 3000
number_of_terms : 3772
total_postings : 15405
average_posting_length : 4.084040296924709


In [77]:
# General Query Processing Function
# دالة عامة لمعالجة الاستعلام

# هذه الدالة تطبق على query نفس خطوات المعالجة التي طبقناها على الوثائق:
# Normalization
# Tokenization
# Stop Words Removal
# Stemming

def process_query(query):
    processed_query = preprocess_stemming(query)
    query_tokens = processed_query.split()
    
    return {
        "original_query": query,
        "processed_query": processed_query,
        "query_tokens": query_tokens
    }

print("process_query function is ready.")

process_query function is ready.


In [78]:
# Test Query Processing
# تجربة معالجة الاستعلام

test_query = "What is the best way to lose weight?"

query_info = process_query(test_query)

print("Original Query:")
print(query_info["original_query"])

print("\nProcessed Query:")
print(query_info["processed_query"])

print("\nQuery Tokens:")
print(query_info["query_tokens"])

Original Query:
What is the best way to lose weight?

Processed Query:
best way lose weight

Query Tokens:
['best', 'way', 'lose', 'weight']


In [79]:
# Query Representation using TF-IDF
# تمثيل الاستعلام باستخدام TF-IDF

# مهم جداً:
# لا نستخدم fit على query
# نستخدم transform فقط لأن vectorizer متعلم مسبقاً من الوثائق
# هذا يجعل query والوثائق في نفس فضاء التمثيل

def represent_query_tfidf(query, tfidf_vectorizer):
    query_info = process_query(query)
    
    query_vector = tfidf_vectorizer.transform([
        query_info["processed_query"]
    ])
    
    return query_vector

print("TF-IDF query representation function is ready.")

TF-IDF query representation function is ready.


In [80]:
# Test TF-IDF Query Representation
# اختبار تمثيل query بنفس تمثيل وثائق TF-IDF

tfidf_query_vector_1 = represent_query_tfidf(
    query=test_query,
    tfidf_vectorizer=tfidf_vectorizer_1
)

tfidf_query_vector_2 = represent_query_tfidf(
    query=test_query,
    tfidf_vectorizer=tfidf_vectorizer_2
)

print("Dataset 1 TF-IDF query vector shape:", tfidf_query_vector_1.shape)
print("Dataset 2 TF-IDF query vector shape:", tfidf_query_vector_2.shape)

print("\nDataset 1 TF-IDF document matrix shape:", tfidf_matrix_1.shape)
print("Dataset 2 TF-IDF document matrix shape:", tfidf_matrix_2.shape)

Dataset 1 TF-IDF query vector shape: (1, 17463)
Dataset 2 TF-IDF query vector shape: (1, 3772)

Dataset 1 TF-IDF document matrix shape: (3000, 17463)
Dataset 2 TF-IDF document matrix shape: (3000, 3772)


In [81]:
    # Query Representation using Word2Vec
# تمثيل الاستعلام باستخدام Word2Vec

# Word2Vec يمثل الكلمات وليس الجملة كاملة
# لذلك نحول query إلى tokens
# ثم نحسب متوسط vectors الكلمات
# وهذا نفس الشيء الذي فعلناه مع الوثائق

def represent_query_word2vec(query, word2vec_model, vector_size=100):
    query_info = process_query(query)
    
    query_vector = document_to_vector(
        query_info["query_tokens"],
        word2vec_model,
        vector_size=vector_size
    )
    
    return query_vector

print("Word2Vec query representation function is ready.")

Word2Vec query representation function is ready.


In [82]:
# Test Word2Vec Query Representation
# اختبار تمثيل query بنفس تمثيل وثائق Word2Vec

word2vec_query_vector_1 = represent_query_word2vec(
    query=test_query,
    word2vec_model=word2vec_model_1,
    vector_size=word2vec_matrix_1.shape[1]
)

word2vec_query_vector_2 = represent_query_word2vec(
    query=test_query,
    word2vec_model=word2vec_model_2,
    vector_size=word2vec_matrix_2.shape[1]
)

print("Dataset 1 Word2Vec query vector shape:", word2vec_query_vector_1.shape)
print("Dataset 2 Word2Vec query vector shape:", word2vec_query_vector_2.shape)

print("\nDataset 1 Word2Vec document matrix shape:", word2vec_matrix_1.shape)
print("Dataset 2 Word2Vec document matrix shape:", word2vec_matrix_2.shape)

print("\nFirst 10 values from Dataset 1 query vector:")
print(word2vec_query_vector_1[:10])

Dataset 1 Word2Vec query vector shape: (100,)
Dataset 2 Word2Vec query vector shape: (100,)

Dataset 1 Word2Vec document matrix shape: (3000, 100)
Dataset 2 Word2Vec document matrix shape: (3000, 100)

First 10 values from Dataset 1 query vector:
[-0.05636612 -0.15701692 -0.00593588 -0.04015846  0.18169954 -0.21450163
  0.03174451  0.35626256 -0.23253241 -0.06862305]


In [83]:
# Query Representation for BM25
# تمثيل الاستعلام من أجل BM25

# BM25 يحتاج query على شكل قائمة كلمات tokens
# وهذه tokens ناتجة من نفس preprocessing المستخدم مع الوثائق

def represent_query_bm25(query):
    query_info = process_query(query)
    return query_info["query_tokens"]

print("BM25 query representation function is ready.")

BM25 query representation function is ready.


In [84]:
# Test BM25 Query Representation
# اختبار تمثيل query من أجل BM25

bm25_query_tokens = represent_query_bm25(test_query)

print("BM25 Query Tokens:")
print(bm25_query_tokens)

print("\nExample document tokens from Dataset 1 BM25 corpus:")
print(tokenized_corpus_1[0][:20])

BM25 Query Tokens:
['best', 'way', 'lose', 'weight']

Example document tokens from Dataset 1 BM25 corpus:
['contracept', 'form', 'high', 'school', 'student', 'oppon', 'forfeit', 'everi', 'round', 'none', 'argument', 'answer', 'like', 'idea', 'win', 'default', 'tule', 'good', 'student', 'get']


In [85]:
# Query Processing for Inverted Index
# معالجة الاستعلام من أجل الفهرس المعكوس

# هذه الدالة تفحص كلمات query داخل الفهرس
# وتعرض هل الكلمة موجودة وكم وثيقة تحتويها

def query_terms_in_index(query, inverted_index):
    query_info = process_query(query)
    
    terms_status = []
    
    for token in query_info["query_tokens"]:
        if token in inverted_index:
            document_frequency = len(inverted_index[token])
        else:
            document_frequency = 0
        
        terms_status.append({
            "query_term": token,
            "exists_in_index": token in inverted_index,
            "document_frequency": document_frequency
        })
    
    return pd.DataFrame(terms_status)

print("Inverted Index query processing function is ready.")

Inverted Index query processing function is ready.


In [86]:
# Test Query Terms in Inverted Index
# اختبار وجود كلمات query داخل الفهرس المعكوس

query_index_status_1 = query_terms_in_index(
    query=test_query,
    inverted_index=inverted_index_1
)

query_index_status_2 = query_terms_in_index(
    query=test_query,
    inverted_index=inverted_index_2
)

print("Query terms in Dataset 1 inverted index:")
display(query_index_status_1)

print("Query terms in Dataset 2 inverted index:")
display(query_index_status_2)

Query terms in Dataset 1 inverted index:


,query_term,exists_in_index,document_frequency
0,best,True,320
1,way,True,713
2,lose,True,139
3,weight,True,50


Query terms in Dataset 2 inverted index:


,query_term,exists_in_index,document_frequency
0,best,True,249
1,way,True,73
2,lose,True,14
3,weight,True,18


In [87]:
# Test Query Terms in Inverted Index
# اختبار وجود كلمات query داخل الفهرس المعكوس

query_index_status_1 = query_terms_in_index(
    query=test_query,
    inverted_index=inverted_index_1
)

query_index_status_2 = query_terms_in_index(
    query=test_query,
    inverted_index=inverted_index_2
)

print("Query terms in Dataset 1 inverted index:")
display(query_index_status_1)

print("Query terms in Dataset 2 inverted index:")
display(query_index_status_2)

Query terms in Dataset 1 inverted index:


,query_term,exists_in_index,document_frequency
0,best,True,320
1,way,True,713
2,lose,True,139
3,weight,True,50


Query terms in Dataset 2 inverted index:


,query_term,exists_in_index,document_frequency
0,best,True,249
1,way,True,73
2,lose,True,14
3,weight,True,18


In [88]:
# Complete Query Representation Function
# دالة تجمع كل تمثيلات query في مكان واحد

# هذه الدالة مفيدة لأنها توضح أن query يمكن تمثيله بنفس طرق تمثيل الوثائق:
# TF-IDF
# Word2Vec
# BM25 Tokens
# Inverted Index Terms

def complete_query_representation(query, dataset_number=1):
    query_info = process_query(query)
    
    if dataset_number == 1:
        tfidf_vectorizer = tfidf_vectorizer_1
        word2vec_model = word2vec_model_1
        word2vec_matrix = word2vec_matrix_1
        inverted_index = inverted_index_1
    else:
        tfidf_vectorizer = tfidf_vectorizer_2
        word2vec_model = word2vec_model_2
        word2vec_matrix = word2vec_matrix_2
        inverted_index = inverted_index_2
    
    tfidf_vector = represent_query_tfidf(
        query=query,
        tfidf_vectorizer=tfidf_vectorizer
    )
    
    word2vec_vector = represent_query_word2vec(
        query=query,
        word2vec_model=word2vec_model,
        vector_size=word2vec_matrix.shape[1]
    )
    
    bm25_tokens = represent_query_bm25(query)
    
    index_status = query_terms_in_index(
        query=query,
        inverted_index=inverted_index
    )
    
    return {
        "query_info": query_info,
        "tfidf_vector": tfidf_vector,
        "word2vec_vector": word2vec_vector,
        "bm25_tokens": bm25_tokens,
        "index_status": index_status
    }

print("complete_query_representation function is ready.")

complete_query_representation function is ready.


In [89]:
# Test Complete Query Representation
# اختبار تمثيل query بشكل كامل على الداتاسيت الأولى والثانية

query_representation_1 = complete_query_representation(
    query=test_query,
    dataset_number=1
)

query_representation_2 = complete_query_representation(
    query=test_query,
    dataset_number=2
)

print("Dataset 1 Query Processing:")
print("Original Query:", query_representation_1["query_info"]["original_query"])
print("Processed Query:", query_representation_1["query_info"]["processed_query"])
print("Tokens:", query_representation_1["query_info"]["query_tokens"])
print("TF-IDF Shape:", query_representation_1["tfidf_vector"].shape)
print("Word2Vec Shape:", query_representation_1["word2vec_vector"].shape)
print("BM25 Tokens:", query_representation_1["bm25_tokens"])

print("\nDataset 1 Index Status:")
display(query_representation_1["index_status"])

print("\n" + "=" * 100)

print("\nDataset 2 Query Processing:")
print("Original Query:", query_representation_2["query_info"]["original_query"])
print("Processed Query:", query_representation_2["query_info"]["processed_query"])
print("Tokens:", query_representation_2["query_info"]["query_tokens"])
print("TF-IDF Shape:", query_representation_2["tfidf_vector"].shape)
print("Word2Vec Shape:", query_representation_2["word2vec_vector"].shape)
print("BM25 Tokens:", query_representation_2["bm25_tokens"])

print("\nDataset 2 Index Status:")
display(query_representation_2["index_status"])

Dataset 1 Query Processing:
Original Query: What is the best way to lose weight?
Processed Query: best way lose weight
Tokens: ['best', 'way', 'lose', 'weight']
TF-IDF Shape: (1, 17463)
Word2Vec Shape: (100,)
BM25 Tokens: ['best', 'way', 'lose', 'weight']

Dataset 1 Index Status:


,query_term,exists_in_index,document_frequency
0,best,True,320
1,way,True,713
2,lose,True,139
3,weight,True,50




Dataset 2 Query Processing:
Original Query: What is the best way to lose weight?
Processed Query: best way lose weight
Tokens: ['best', 'way', 'lose', 'weight']
TF-IDF Shape: (1, 3772)
Word2Vec Shape: (100,)
BM25 Tokens: ['best', 'way', 'lose', 'weight']

Dataset 2 Index Status:


,query_term,exists_in_index,document_frequency
0,best,True,249
1,way,True,73
2,lose,True,14
3,weight,True,18


In [90]:
# Compare Query and Document Representation
# مقارنة تمثيل query مع تمثيل الوثائق

# الهدف أن نثبت أن query والوثائق تمثلا بنفس الطريقة

print("TF-IDF Comparison:")
print("Query TF-IDF shape:", query_representation_1["tfidf_vector"].shape)
print("Documents TF-IDF matrix shape:", tfidf_matrix_1.shape)

print("\nWord2Vec Comparison:")
print("Query Word2Vec shape:", query_representation_1["word2vec_vector"].shape)
print("Documents Word2Vec matrix shape:", word2vec_matrix_1.shape)

print("\nBM25 Comparison:")
print("Query BM25 tokens:", query_representation_1["bm25_tokens"])
print("Example document BM25 tokens:", tokenized_corpus_1[0][:20])

print("\nText Processing Comparison:")
print("Sample document processed text:")
print(work1_df.iloc[0]["processed_text"][:300])

print("\nProcessed query:")
print(query_representation_1["query_info"]["processed_query"])

TF-IDF Comparison:
Query TF-IDF shape: (1, 17463)
Documents TF-IDF matrix shape: (3000, 17463)

Word2Vec Comparison:
Query Word2Vec shape: (100,)
Documents Word2Vec matrix shape: (3000, 100)

BM25 Comparison:
Query BM25 tokens: ['best', 'way', 'lose', 'weight']
Example document BM25 tokens: ['contracept', 'form', 'high', 'school', 'student', 'oppon', 'forfeit', 'everi', 'round', 'none', 'argument', 'answer', 'like', 'idea', 'win', 'default', 'tule', 'good', 'student', 'get']

Text Processing Comparison:
Sample document processed text:
contracept form high school student oppon forfeit everi round none argument answer like idea win default tule good student get involv address big issu like teen pregnanc need abl answer argument like mine simpli prepar abstin type respons also awar condom may sold minor state retail say illeg sell f

Processed query:
best way lose weight


In [93]:
# Query Processing Summary Table
# جدول يلخص كيف تمت معالجة وتمثيل الاستعلام باللغة الإنجليزية

summary_data = [
    {
        "Step": "Preprocessing",
        "Query Output": "Processed query and query tokens",
        "Same Method as Documents": "Yes",
        "Explanation": "The query is processed using the same preprocessing function used for documents: normalization, tokenization, stop word removal, and stemming."
    },
    {
        "Step": "TF-IDF Representation",
        "Query Output": str(query_representation_1["tfidf_vector"].shape),
        "Same Method as Documents": "Yes",
        "Explanation": "The query is transformed using the same TF-IDF vectorizer fitted on the documents. We use transform only, not fit."
    },
    {
        "Step": "Word2Vec Representation",
        "Query Output": str(query_representation_1["word2vec_vector"].shape),
        "Same Method as Documents": "Yes",
        "Explanation": "The query is represented by averaging the Word2Vec vectors of its tokens, using the same method applied to documents."
    },
    {
        "Step": "BM25 Representation",
        "Query Output": str(query_representation_1["bm25_tokens"]),
        "Same Method as Documents": "Yes",
        "Explanation": "The query is converted into tokens using the same preprocessing pipeline used to build the BM25 tokenized corpus."
    },
    {
        "Step": "Inverted Index Lookup",
        "Query Output": "Processed query terms",
        "Same Method as Documents": "Yes",
        "Explanation": "The processed query terms are searched inside the inverted index built from the processed document texts."
    }
]

query_processing_summary_df = pd.DataFrame(summary_data)

display(query_processing_summary_df)

,Step,Query Output,Same Method as Documents,Explanation
0,Preprocessing,Processed query and query tokens,Yes,The query is processed using the same preproce...
1,TF-IDF Representation,"(1, 17463)",Yes,The query is transformed using the same TF-IDF...
2,Word2Vec Representation,"(100,)",Yes,The query is represented by averaging the Word...
3,BM25 Representation,"['best', 'way', 'lose', 'weight']",Yes,The query is converted into tokens using the s...
4,Inverted Index Lookup,Processed query terms,Yes,The processed query terms are searched inside ...


In [94]:
# Save Query Processing Results
# حفظ نتائج Query Processing

import os
import pickle

SAVE_DIR = "ir_project_saved_files"
os.makedirs(SAVE_DIR, exist_ok=True)

# حفظ جدول الملخص باللغة الإنجليزية

query_processing_summary_df.to_csv(
    os.path.join(SAVE_DIR, "query_processing_summary_english.csv"),
    index=False
)

# حفظ حالة كلمات query داخل الفهرس للداتاسيت الأولى

query_index_status_1.to_csv(
    os.path.join(SAVE_DIR, "query_terms_in_index_dataset1.csv"),
    index=False
)

# حفظ حالة كلمات query داخل الفهرس للداتاسيت الثانية

query_index_status_2.to_csv(
    os.path.join(SAVE_DIR, "query_terms_in_index_dataset2.csv"),
    index=False
)

# حفظ تقرير مختصر عن تمثيل query
# لا نحفظ vectors نفسها لأنها ممكن تكون كبيرة
# نحفظ فقط الأشكال والمعلومات المهمة للمناقشة

query_processing_report = {
    "original_query": query_info["original_query"],
    "processed_query": query_info["processed_query"],
    "query_tokens": query_info["query_tokens"],
    
    "tfidf_query_vector_shape_dataset1": query_representation_1["tfidf_vector"].shape,
    "tfidf_query_vector_shape_dataset2": query_representation_2["tfidf_vector"].shape,
    
    "word2vec_query_vector_shape_dataset1": query_representation_1["word2vec_vector"].shape,
    "word2vec_query_vector_shape_dataset2": query_representation_2["word2vec_vector"].shape,
    
    "bm25_query_tokens_dataset1": query_representation_1["bm25_tokens"],
    "bm25_query_tokens_dataset2": query_representation_2["bm25_tokens"]
}

with open(os.path.join(SAVE_DIR, "query_processing_report.pkl"), "wb") as f:
    pickle.dump(query_processing_report, f)

print("Query Processing results saved successfully.")
print("Save folder:", SAVE_DIR)

Query Processing results saved successfully.
Save folder: ir_project_saved_files


In [95]:
# Check Saved Query Processing Files
# التحقق من حفظ ملفات Query Processing

import os

SAVE_DIR = "ir_project_saved_files"

query_processing_files = [
    "query_processing_summary_english.csv",
    "query_terms_in_index_dataset1.csv",
    "query_terms_in_index_dataset2.csv",
    "query_processing_report.pkl"
]

for file_name in query_processing_files:
    path = os.path.join(SAVE_DIR, file_name)
    
    if os.path.exists(path):
        size_kb = os.path.getsize(path) / 1024
        print("FOUND:", file_name, "| Size:", round(size_kb, 2), "KB")
    else:
        print("MISSING:", file_name)

FOUND: query_processing_summary_english.csv | Size: 0.88 KB
FOUND: query_terms_in_index_dataset1.csv | Size: 0.1 KB
FOUND: query_terms_in_index_dataset2.csv | Size: 0.1 KB
FOUND: query_processing_report.pkl | Size: 0.45 KB


In [96]:
# Save everything up to Requirement 4
# حفظ كل الملفات المهمة حتى نهاية الطلب الرابع
# الهدف: عدم إعادة التحميل والمعالجة والتدريب من الصفر عند فتح النوتبوك لاحقاً

import os
import pickle
import numpy as np

SAVE_DIR = "ir_project_saved_files"
os.makedirs(SAVE_DIR, exist_ok=True)


def save_dataframe_if_exists(var_name, file_name):
    if var_name in globals():
        globals()[var_name].to_csv(os.path.join(SAVE_DIR, file_name), index=False)
        print("Saved:", file_name)
    else:
        print("Missing:", var_name)


def save_pickle_if_exists(var_name, file_name):
    if var_name in globals():
        with open(os.path.join(SAVE_DIR, file_name), "wb") as f:
            pickle.dump(globals()[var_name], f)
        print("Saved:", file_name)
    else:
        print("Missing:", var_name)


def save_numpy_if_exists(var_name, file_name):
    if var_name in globals():
        np.save(os.path.join(SAVE_DIR, file_name), globals()[var_name])
        print("Saved:", file_name)
    else:
        print("Missing:", var_name)


def save_word2vec_if_exists(var_name, file_name):
    if var_name in globals():
        globals()[var_name].save(os.path.join(SAVE_DIR, file_name))
        print("Saved:", file_name)
    else:
        print("Missing:", var_name)


print("=" * 80)
print("Saving datasets and processed data")
print("=" * 80)

save_dataframe_if_exists("docs1_df", "dataset1_processed_docs.csv")
save_dataframe_if_exists("docs2_df", "dataset2_processed_docs.csv")
save_dataframe_if_exists("work1_df", "work_dataset1.csv")
save_dataframe_if_exists("work2_df", "work_dataset2.csv")


print("\n" + "=" * 80)
print("Saving TF-IDF representation")
print("=" * 80)

save_pickle_if_exists("tfidf_vectorizer_1", "tfidf_vectorizer_1.pkl")
save_pickle_if_exists("tfidf_vectorizer_2", "tfidf_vectorizer_2.pkl")
save_pickle_if_exists("tfidf_matrix_1", "tfidf_matrix_1.pkl")
save_pickle_if_exists("tfidf_matrix_2", "tfidf_matrix_2.pkl")


print("\n" + "=" * 80)
print("Saving Word2Vec representation")
print("=" * 80)

save_word2vec_if_exists("word2vec_model_1", "word2vec_model_dataset1.model")
save_word2vec_if_exists("word2vec_model_2", "word2vec_model_dataset2.model")

save_numpy_if_exists("word2vec_matrix_1", "word2vec_matrix_1.npy")
save_numpy_if_exists("word2vec_matrix_2", "word2vec_matrix_2.npy")


print("\n" + "=" * 80)
print("Saving BM25 representation")
print("=" * 80)

save_pickle_if_exists("tokenized_corpus_1", "tokenized_corpus_1.pkl")
save_pickle_if_exists("tokenized_corpus_2", "tokenized_corpus_2.pkl")


print("\n" + "=" * 80)
print("Saving Hybrid representation results")
print("=" * 80)

save_dataframe_if_exists("serial_results_1", "serial_hybrid_results_dataset1.csv")
save_dataframe_if_exists("serial_results_2", "serial_hybrid_results_dataset2.csv")

save_dataframe_if_exists("parallel_results_1", "parallel_hybrid_results_dataset1.csv")
save_dataframe_if_exists("parallel_results_2", "parallel_hybrid_results_dataset2.csv")

save_dataframe_if_exists("parallel_alpha_08", "parallel_hybrid_alpha_08.csv")
save_dataframe_if_exists("parallel_alpha_03", "parallel_hybrid_alpha_03.csv")


print("\n" + "=" * 80)
print("Saving Indexing results")
print("=" * 80)

save_pickle_if_exists("inverted_index_1", "inverted_index_1.pkl")
save_pickle_if_exists("inverted_index_2", "inverted_index_2.pkl")

save_dataframe_if_exists("index_results_1", "inverted_index_results_dataset1.csv")
save_dataframe_if_exists("index_results_2", "inverted_index_results_dataset2.csv")

save_pickle_if_exists("index_stats_1", "index_stats_1.pkl")
save_pickle_if_exists("index_stats_2", "index_stats_2.pkl")


print("\n" + "=" * 80)
print("Saving Query Processing results")
print("=" * 80)

save_dataframe_if_exists("query_processing_summary_df", "query_processing_summary_english.csv")
save_dataframe_if_exists("query_index_status_1", "query_terms_in_index_dataset1.csv")
save_dataframe_if_exists("query_index_status_2", "query_terms_in_index_dataset2.csv")

if "query_representation_1" in globals() and "query_representation_2" in globals():
    query_processing_report = {
        "dataset1_original_query": query_representation_1["query_info"]["original_query"],
        "dataset1_processed_query": query_representation_1["query_info"]["processed_query"],
        "dataset1_query_tokens": query_representation_1["query_info"]["query_tokens"],
        "dataset1_tfidf_shape": query_representation_1["tfidf_vector"].shape,
        "dataset1_word2vec_shape": query_representation_1["word2vec_vector"].shape,
        "dataset1_bm25_tokens": query_representation_1["bm25_tokens"],

        "dataset2_original_query": query_representation_2["query_info"]["original_query"],
        "dataset2_processed_query": query_representation_2["query_info"]["processed_query"],
        "dataset2_query_tokens": query_representation_2["query_info"]["query_tokens"],
        "dataset2_tfidf_shape": query_representation_2["tfidf_vector"].shape,
        "dataset2_word2vec_shape": query_representation_2["word2vec_vector"].shape,
        "dataset2_bm25_tokens": query_representation_2["bm25_tokens"]
    }

    with open(os.path.join(SAVE_DIR, "query_processing_report.pkl"), "wb") as f:
        pickle.dump(query_processing_report, f)

    print("Saved: query_processing_report.pkl")
else:
    print("Missing: query_representation_1 or query_representation_2")


print("\n" + "=" * 80)
print("Saving main search results")
print("=" * 80)

save_dataframe_if_exists("tfidf_results_1", "tfidf_results_dataset1.csv")
save_dataframe_if_exists("tfidf_results_2", "tfidf_results_dataset2.csv")

save_dataframe_if_exists("word2vec_results_1", "word2vec_results_dataset1.csv")
save_dataframe_if_exists("word2vec_results_2", "word2vec_results_dataset2.csv")

save_dataframe_if_exists("bm25_results_1", "bm25_results_dataset1.csv")
save_dataframe_if_exists("bm25_results_2", "bm25_results_dataset2.csv")
save_dataframe_if_exists("bm25_results_changed_params", "bm25_results_changed_params.csv")


print("\n" + "=" * 80)
print("Saving completed.")
print("Save folder:", SAVE_DIR)
print("=" * 80)

Saving datasets and processed data
Saved: dataset1_processed_docs.csv
Saved: dataset2_processed_docs.csv
Saved: work_dataset1.csv
Saved: work_dataset2.csv

Saving TF-IDF representation
Saved: tfidf_vectorizer_1.pkl
Saved: tfidf_vectorizer_2.pkl
Saved: tfidf_matrix_1.pkl
Saved: tfidf_matrix_2.pkl

Saving Word2Vec representation
Saved: word2vec_model_dataset1.model
Saved: word2vec_model_dataset2.model
Saved: word2vec_matrix_1.npy
Saved: word2vec_matrix_2.npy

Saving BM25 representation
Saved: tokenized_corpus_1.pkl
Saved: tokenized_corpus_2.pkl

Saving Hybrid representation results
Saved: serial_hybrid_results_dataset1.csv
Saved: serial_hybrid_results_dataset2.csv
Saved: parallel_hybrid_results_dataset1.csv
Saved: parallel_hybrid_results_dataset2.csv
Saved: parallel_hybrid_alpha_08.csv
Saved: parallel_hybrid_alpha_03.csv

Saving Indexing results
Saved: inverted_index_1.pkl
Saved: inverted_index_2.pkl
Saved: inverted_index_results_dataset1.csv
Saved: inverted_index_results_dataset2.csv
Sa

In [5]:
# Query Refinement Setup
# إعداد القواميس والمتغيرات اللازمة لتحسين الاستعلام

# هذا القاموس يستخدم لتصحيح بعض الأخطاء الإملائية الشائعة
# وضعنا كلمات بسيطة كمثال عملي للمشروع

spelling_corrections = {
    "weigth": "weight",
    "wieght": "weight",
    "wheight": "weight",
    "loose": "lose",
    "losingg": "losing",
    "helth": "health",
    "excersise": "exercise",
    "programing": "programming",
    "lern": "learn",
    "lerning": "learning",
    "tecnology": "technology",
    "enviroment": "environment"
}


# هذا القاموس يستخدم لإضافة مرادفات أو كلمات مرتبطة بالمعنى
# الهدف هو توسيع query حتى تزيد فرصة العثور على وثائق مناسبة

synonym_dictionary = {
    "weight": ["loss", "diet", "fitness"],
    "lose": ["weight", "loss", "diet"],
    "health": ["medical", "fitness", "wellness"],
    "exercise": ["workout", "fitness", "training"],
    "programming": ["coding", "software", "development"],
    "learn": ["study", "education", "training"],
    "car": ["automobile", "vehicle"],
    "climate": ["environment", "weather"],
    "movie": ["film", "cinema"],
    "food": ["meal", "diet", "nutrition"]
}


# هذا يمثل سجل بحث بسيط للمستخدم
# لاحقاً في موقع React يمكن تخزينه في database أو local storage

user_search_history = [
    "weight loss diet",
    "best exercise for fitness",
    "healthy food and nutrition",
    "learn programming basics"
]

print("Query refinement setup is ready.")

Query refinement setup is ready.


In [6]:
# Spelling Correction
# دالة لتصحيح الأخطاء الإملائية البسيطة داخل query

def correct_query_spelling(query):
    query_info = process_query(query)
    tokens = query_info["query_tokens"]
    
    corrected_tokens = []
    
    for token in tokens:
        if token in spelling_corrections:
            corrected_tokens.append(spelling_corrections[token])
        else:
            corrected_tokens.append(token)
    
    corrected_query = " ".join(corrected_tokens)
    
    return corrected_query


print("Spelling correction function is ready.")

Spelling correction function is ready.


In [7]:
# Query Expansion
# دالة لتوسيع query بإضافة مرادفات وكلمات مرتبطة

def expand_query_with_synonyms(query, max_synonyms_per_term=2):
    query_info = process_query(query)
    tokens = query_info["query_tokens"]
    
    expanded_tokens = []
    
    for token in tokens:
        expanded_tokens.append(token)
        
        if token in synonym_dictionary:
            related_terms = synonym_dictionary[token][:max_synonyms_per_term]
            
            for related_term in related_terms:
                if related_term not in expanded_tokens:
                    expanded_tokens.append(related_term)
    
    expanded_query = " ".join(expanded_tokens)
    
    return expanded_query


print("Query expansion function is ready.")

Query expansion function is ready.


In [8]:
# Query Suggestion from Search History
# اقتراح استعلام قريب اعتماداً على سجل البحث السابق

def suggest_query_from_history(query, search_history, top_k=3):
    processed_query = preprocess_stemming(query)
    query_tokens = set(processed_query.split())
    
    suggestions = []
    
    for old_query in search_history:
        processed_old_query = preprocess_stemming(old_query)
        old_query_tokens = set(processed_old_query.split())
        
        common_terms = query_tokens.intersection(old_query_tokens)
        
        score = len(common_terms)
        
        if score > 0:
            suggestions.append({
                "suggested_query": old_query,
                "similarity_score": score
            })
    
    suggestions = sorted(
        suggestions,
        key=lambda x: x["similarity_score"],
        reverse=True
    )
    
    return pd.DataFrame(suggestions[:top_k])


print("Query suggestion function is ready.")

Query suggestion function is ready.


In [9]:
# Full Query Refinement
# دالة كاملة تجمع التصحيح والتوسيع والاقتراح

def refine_query(query, use_spelling=True, use_expansion=True, use_history=True):
    original_query = query
    
    # الخطوة الأولى: معالجة query مبدئياً
    
    processed_original = process_query(original_query)["processed_query"]
    
    # الخطوة الثانية: تصحيح الأخطاء الإملائية
    
    if use_spelling:
        corrected_query = correct_query_spelling(original_query)
    else:
        corrected_query = processed_original
    
    # الخطوة الثالثة: توسيع query بإضافة مرادفات
    
    if use_expansion:
        expanded_query = expand_query_with_synonyms(corrected_query)
    else:
        expanded_query = corrected_query
    
    # الخطوة الرابعة: اقتراح queries من سجل البحث
    
    if use_history:
        history_suggestions = suggest_query_from_history(
            query=expanded_query,
            search_history=user_search_history,
            top_k=3
        )
    else:
        history_suggestions = pd.DataFrame()
    
    refined_query = expanded_query
    
    refinement_report = {
        "original_query": original_query,
        "processed_original_query": processed_original,
        "corrected_query": corrected_query,
        "expanded_query": expanded_query,
        "refined_query": refined_query,
        "history_suggestions": history_suggestions
    }
    
    return refinement_report


print("Full query refinement function is ready.")

Full query refinement function is ready.


In [10]:
# Test Query Refinement
# اختبار تحسين query

test_refinement_query = "weigth loss"

refinement_report = refine_query(
    query=test_refinement_query,
    use_spelling=True,
    use_expansion=True,
    use_history=True
)

print("Original Query:")
print(refinement_report["original_query"])

print("\nProcessed Original Query:")
print(refinement_report["processed_original_query"])

print("\nCorrected Query:")
print(refinement_report["corrected_query"])

print("\nExpanded Query:")
print(refinement_report["expanded_query"])

print("\nFinal Refined Query:")
print(refinement_report["refined_query"])

print("\nQuery Suggestions from History:")
display(refinement_report["history_suggestions"])

Original Query:
weigth loss

Processed Original Query:
weigth loss

Corrected Query:
weight loss

Expanded Query:
weight loss diet loss

Final Refined Query:
weight loss diet loss

Query Suggestions from History:


,suggested_query,similarity_score
0,weight loss diet,3


In [11]:
# Compare Search Before and After Query Refinement
# مقارنة نتائج البحث قبل وبعد تحسين query

original_query = refinement_report["original_query"]
refined_query = refinement_report["refined_query"]

print("Original Query:")
print(original_query)

print("\nRefined Query:")
print(refined_query)


print("\n" + "=" * 100)
print("BM25 Results Before Query Refinement")
print("=" * 100)

bm25_before_refinement = search_bm25(
    query=original_query,
    docs_df=work1_df,
    tokenized_corpus=tokenized_corpus_1,
    top_k=5,
    k1=1.5,
    b=0.75
)

display(bm25_before_refinement)


print("\n" + "=" * 100)
print("BM25 Results After Query Refinement")
print("=" * 100)

bm25_after_refinement = search_bm25(
    query=refined_query,
    docs_df=work1_df,
    tokenized_corpus=tokenized_corpus_1,
    top_k=5,
    k1=1.5,
    b=0.75
)

display(bm25_after_refinement)

Original Query:
weigth loss

Refined Query:
weight loss diet loss

BM25 Results Before Query Refinement


,doc_id,text,bm25_score,k1,b
540,4c34b2a1-2019-04-18T13:21:19Z-00005-000,"We should change the date of Australia Day, as...",7.671209,1.5,0.75
706,faf4daa-2019-04-18T12:54:20Z-00003-000,The Earth is Young Evidence Consistent With a ...,7.230933,1.5,0.75
318,19540073-2019-04-18T13:55:46Z-00005-000,Necrophilia Should be Legalized Resolution is ...,6.266845,1.5,0.75
2157,544f0063-2019-04-18T19:56:01Z-00001-000,"Wet Feet Dry Feet Act of 1994 You siad that ""S...",6.124305,1.5,0.75
2071,49d899f-2019-04-18T16:58:51Z-00003-000,Legalization of Marjuana I accept but to frame...,6.096571,1.5,0.75



BM25 Results After Query Refinement


,doc_id,text,bm25_score,k1,b
540,4c34b2a1-2019-04-18T13:21:19Z-00005-000,"We should change the date of Australia Day, as...",15.342417,1.5,0.75
706,faf4daa-2019-04-18T12:54:20Z-00003-000,The Earth is Young Evidence Consistent With a ...,14.461867,1.5,0.75
2412,f4eba7e2-2019-04-18T12:53:04Z-00002-000,men have become tools of their tools Even thou...,14.262384,1.5,0.75
318,19540073-2019-04-18T13:55:46Z-00005-000,Necrophilia Should be Legalized Resolution is ...,12.533690,1.5,0.75
2157,544f0063-2019-04-18T19:56:01Z-00001-000,"Wet Feet Dry Feet Act of 1994 You siad that ""S...",12.248609,1.5,0.75


In [20]:
# دالة عامة لترتيب النتائج حسب الدرجة
# هذه الدالة تأخذ جدول نتائج واسم عمود الدرجة
# ثم ترتب الوثائق من أعلى درجة إلى أقل درجة

def rank_results_by_score(results_df, score_column, top_k=10):
    ranked_results = results_df.sort_values(
        by=score_column,
        ascending=False
    ).head(top_k).copy()
    
    ranked_results["rank"] = range(1, len(ranked_results) + 1)
    
    columns_order = ["rank", "doc_id", score_column, "text"]
    
    available_columns = [
        column
        for column in columns_order
        if column in ranked_results.columns
    ]
    
    return ranked_results[available_columns]

In [21]:
# مطابقة الاستعلام مع الوثائق باستخدام طريقة التردد والوزن
# نحول الاستعلام إلى متجه بنفس فضاء الوثائق
# ثم نحسب التشابه بين متجه الاستعلام ومتجهات الوثائق
# بعدها نرتب النتائج حسب أعلى درجة تشابه

def match_and_rank_tfidf(query, docs_df, vectorizer, tfidf_matrix, top_k=10):
    processed_query = preprocess_stemming(query)
    
    query_vector = vectorizer.transform([processed_query])
    
    similarity_scores = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()
    
    results = docs_df[["doc_id", "text"]].copy()
    results["similarity_score"] = similarity_scores
    
    ranked_results = rank_results_by_score(
        results_df=results,
        score_column="similarity_score",
        top_k=top_k
    )
    
    return ranked_results

In [27]:
# اختبار المطابقة والترتيب باستخدام طريقة التردد والوزن

test_query = "what is the best way to lose weight"

tfidf_matching_ranking_1 = match_and_rank_tfidf(
    query=test_query,
    docs_df=work1_df,
    vectorizer=tfidf_vectorizer_1,
    tfidf_matrix=tfidf_matrix_1,
    top_k=5
)

tfidf_matching_ranking_2 = match_and_rank_tfidf(
    query=test_query,
    docs_df=work2_df,
    vectorizer=tfidf_vectorizer_2,
    tfidf_matrix=tfidf_matrix_2,
    top_k=5
)

print("TF_IDF results for dataset 1")
display(tfidf_matching_ranking_1)

print("TF_IDF results for dataset 2")
display(tfidf_matching_ranking_2)

TF_IDF results for dataset 1


,rank,doc_id,similarity_score,text
1338,1,a88e4c42-2019-04-18T19:40:07Z-00005-000,0.374399,I (im_always_right) will lose this debate if I...
1336,2,a88e4c42-2019-04-18T19:40:07Z-00003-000,0.359070,I (im_always_right) will lose this debate Oh I...
1337,3,a88e4c42-2019-04-18T19:40:07Z-00004-000,0.283284,I (im_always_right) will lose this debate I ne...
30,4,d6517702-2019-04-18T12:36:24Z-00001-000,0.262720,Science is the best! Science is the best!
1334,5,a88e4c42-2019-04-18T19:40:07Z-00001-000,0.188869,I (im_always_right) will lose this debate This...


TF_IDF results for dataset 2


,rank,doc_id,similarity_score,text
2478,1,2559,1.000000,What are the best ways to lose weight?
2623,2,2711,0.715545,What is the best method of losing weight?
2477,3,2558,0.602135,Why do I not lose weight when I throw up?
2624,4,2712,0.584073,What are the best way of loose the weight?
1633,5,1693,0.564573,How do I lose weight without doing any sport?


In [23]:
# مطابقة الاستعلام مع الوثائق باستخدام المتجهات الدلالية
# نحول الاستعلام إلى متجه دلالي
# ثم نحسب التشابه بين متجه الاستعلام ومتجهات الوثائق
# بعدها نرتب النتائج حسب أعلى درجة تشابه

def match_and_rank_word2vec(query, docs_df, word2vec_model, word2vec_matrix, top_k=10):
    processed_query = preprocess_stemming(query)
    query_tokens = processed_query.split()
    
    query_vector = document_to_vector(
        query_tokens,
        word2vec_model,
        vector_size=word2vec_matrix.shape[1]
    ).reshape(1, -1)
    
    similarity_scores = cosine_similarity(
        query_vector,
        word2vec_matrix
    ).flatten()
    
    results = docs_df[["doc_id", "text"]].copy()
    results["similarity_score"] = similarity_scores
    
    ranked_results = rank_results_by_score(
        results_df=results,
        score_column="similarity_score",
        top_k=top_k
    )
    
    return ranked_results

In [26]:
# اختبار المطابقة والترتيب باستخدام المتجهات الدلالية

word2vec_matching_ranking_1 = match_and_rank_word2vec(
    query=test_query,
    docs_df=work1_df,
    word2vec_model=word2vec_model_1,
    word2vec_matrix=word2vec_matrix_1,
    top_k=5
)

word2vec_matching_ranking_2 = match_and_rank_word2vec(
    query=test_query,
    docs_df=work2_df,
    word2vec_model=word2vec_model_2,
    word2vec_matrix=word2vec_matrix_2,
    top_k=5
)

print("Word2Vec results for dataset1")
display(word2vec_matching_ranking_1)

print("Word2Vec results for dataset2")
display(word2vec_matching_ranking_2)

Word2Vec results for dataset1


,rank,doc_id,similarity_score,text
148,1,197ac9b-2019-04-18T19:47:57Z-00001-000,0.824176,Baseball is a better sport than basketball. Ag...
1336,2,a88e4c42-2019-04-18T19:40:07Z-00003-000,0.824026,I (im_always_right) will lose this debate Oh I...
65,3,a60d2aa5-2019-04-18T17:14:53Z-00002-000,0.823416,Russell Hantz Should Have Won Survivor: Samoa ...
147,4,197ac9b-2019-04-18T19:47:57Z-00000-000,0.808622,Baseball is a better sport than basketball. Hi...
64,5,a60d2aa5-2019-04-18T17:14:53Z-00001-000,0.807318,Russell Hantz Should Have Won Survivor: Samoa ...


Word2Vec results for dataset2


,rank,doc_id,similarity_score,text
2478,1,2559,1.000000,What are the best ways to lose weight?
2624,2,2712,0.999821,What are the best way of loose the weight?
2623,3,2711,0.999735,What is the best method of losing weight?
1853,4,1919,0.999579,How could I gain weight in a healthy way?
2360,5,2441,0.999565,What is the best way to say 1?


In [28]:
# مطابقة الاستعلام مع الوثائق باستخدام نموذج الاحتمالية
# هذه الطريقة تستخدم درجات النموذج الاحتمالي الجاهزة
# ثم نرتب النتائج حسب أعلى درجة

def match_and_rank_bm25(query, docs_df, tokenized_corpus, top_k=10, k1=1.5, b=0.75):
    results = search_bm25(
        query=query,
        docs_df=docs_df,
        tokenized_corpus=tokenized_corpus,
        top_k=len(docs_df),
        k1=k1,
        b=b
    )
    
    ranked_results = rank_results_by_score(
        results_df=results,
        score_column="bm25_score",
        top_k=top_k
    )
    
    return ranked_results

In [29]:
# اختبار المطابقة والترتيب باستخدام نموذج الاحتمالية

bm25_matching_ranking_1 = match_and_rank_bm25(
    query=test_query,
    docs_df=work1_df,
    tokenized_corpus=tokenized_corpus_1,
    top_k=5,
    k1=1.5,
    b=0.75
)

bm25_matching_ranking_2 = match_and_rank_bm25(
    query=test_query,
    docs_df=work2_df,
    tokenized_corpus=tokenized_corpus_2,
    top_k=5,
    k1=1.5,
    b=0.75
)

print("BM25 results for data set 1")
display(bm25_matching_ranking_1)

print("BM25 results for data set 2")
display(bm25_matching_ranking_2)

BM25 results for data set 1


,rank,doc_id,bm25_score,text
1336,1,a88e4c42-2019-04-18T19:40:07Z-00003-000,8.370463,I (im_always_right) will lose this debate Oh I...
1061,2,2c5282d9-2019-04-18T19:39:34Z-00001-000,7.876333,Does he fit the definition of a terorist? Note...
512,3,f2f66016-2019-04-18T16:48:35Z-00002-000,7.796105,Pokemon is freakin creepy. You disregard the k...
1047,4,8840debf-2019-04-18T13:05:37Z-00004-000,7.373301,swimming is a better sport than soccer So my o...
65,5,a60d2aa5-2019-04-18T17:14:53Z-00002-000,7.096017,Russell Hantz Should Have Won Survivor: Samoa ...


BM25 results for data set 2


,rank,doc_id,bm25_score,text
2478,1,2559,18.560818,What are the best ways to lose weight?
2623,2,2711,14.414871,What is the best method of losing weight?
2477,3,2558,12.947588,Why do I not lose weight when I throw up?
2624,4,2712,12.566060,What are the best way of loose the weight?
2225,5,2303,11.713884,How do I lose weight and gain muscle?


In [34]:
# مطابقة وترتيب النتائج باستخدام الفهرس المعكوس
# هذه النسخة تصلح مشكلة اختلاف نوع معرف الوثيقة
# نحول معرفات الوثائق في الجدول والفهرس إلى نصوص قبل المطابقة

def match_and_rank_inverted_index(query, docs_df, inverted_index, top_k=10):
    processed_query = preprocess_stemming(query)
    query_tokens = processed_query.split()
    
    document_scores = {}
    
    for token in query_tokens:
        if token in inverted_index:
            posting_list = inverted_index[token]
            
            for doc_id, frequency in posting_list.items():
                doc_id = str(doc_id)
                
                if doc_id not in document_scores:
                    document_scores[doc_id] = 0
                
                document_scores[doc_id] += frequency
    
    if len(document_scores) == 0:
        return pd.DataFrame([
            {
                "rank": "-",
                "doc_id": "-",
                "matching_score": 0,
                "text": "لا توجد وثائق مطابقة لكلمات الاستعلام داخل الفهرس الحالي"
            }
        ])
    
    temp_docs_df = docs_df.copy()
    temp_docs_df["doc_id"] = temp_docs_df["doc_id"].astype(str)
    
    results = temp_docs_df[
        temp_docs_df["doc_id"].isin(document_scores.keys())
    ][["doc_id", "text"]].copy()
    
    results["matching_score"] = results["doc_id"].map(document_scores)
    
    results = results.sort_values(
        by="matching_score",
        ascending=False
    ).head(top_k).copy()
    
    results["rank"] = range(1, len(results) + 1)
    
    results = results[["rank", "doc_id", "matching_score", "text"]]
    
    return results

In [36]:
# إعادة اختبار الفهرس المعكوس على الداتاسيتين

index_matching_ranking_1 = match_and_rank_inverted_index(
    query=test_query,
    docs_df=work1_df,
    inverted_index=inverted_index_1,
    top_k=5
)

index_matching_ranking_2 = match_and_rank_inverted_index(
    query=test_query,
    docs_df=work2_df,
    inverted_index=inverted_index_2,
    top_k=5
)

print("Inverted Index results for dataset 1")
display(index_matching_ranking_1)

print("Inverted Index results for dataset 2")
display(index_matching_ranking_2)

Inverted Index results for dataset 1


,rank,doc_id,matching_score,text
1337,1,a88e4c42-2019-04-18T19:40:07Z-00004-000,12,I (im_always_right) will lose this debate I ne...
612,2,ef778452-2019-04-18T15:25:55Z-00006-000,11,Evolution vs creationism I will support creati...
125,3,a617b5f8-2019-04-18T12:37:07Z-00002-000,11,The US Constitution Should Be a Living Documen...
409,4,939a10f1-2019-04-18T17:47:16Z-00001-000,10,It should not be acceptable for a woman to be ...
2606,5,f7c26c8f-2019-04-18T19:52:26Z-00005-000,10,You should always be patriotic. [Note: I am no...


Inverted Index results for dataset 2


,rank,doc_id,matching_score,text
2478,1,2559,4,What are the best ways to lose weight?
2623,2,2711,3,What is the best method of losing weight?
2811,3,2906,3,What is the best place to visit in the world? ...
396,4,406,3,What is the best way to start robotics? Which ...
2624,5,2712,3,What are the best way of loose the weight?


In [38]:
# مطابقة وترتيب النتائج باستخدام التمثيل الهجين التسلسلي
# في هذه الطريقة نستخدم نموذج الاحتمالية أولاً لجلب وثائق مرشحة
# ثم نستخدم المتجهات الدلالية لإعادة ترتيب الوثائق المرشحة

serial_matching_ranking_1 = serial_hybrid_search(
    query=test_query,
    docs_df=work1_df,
    tokenized_corpus=tokenized_corpus_1,
    word2vec_model=word2vec_model_1,
    word2vec_matrix=word2vec_matrix_1,
    top_k=5,
    candidate_k=100,
    k1=1.5,
    b=0.75
)

serial_matching_ranking_2 = serial_hybrid_search(
    query=test_query,
    docs_df=work2_df,
    tokenized_corpus=tokenized_corpus_2,
    word2vec_model=word2vec_model_2,
    word2vec_matrix=word2vec_matrix_2,
    top_k=5,
    candidate_k=100,
    k1=1.5,
    b=0.75
)

print("Serial Hyprid Results for dataset 1")
display(serial_matching_ranking_1)

print("Serial Hyprid Results for dataset 2")
display(serial_matching_ranking_2)

Serial Hyprid Results for dataset 1


,doc_id,text,bm25_candidate_score,word2vec_rerank_score,candidate_k,k1,b
1336,a88e4c42-2019-04-18T19:40:07Z-00003-000,I (im_always_right) will lose this debate Oh I...,8.370463,0.824026,100,1.5,0.75
65,a60d2aa5-2019-04-18T17:14:53Z-00002-000,Russell Hantz Should Have Won Survivor: Samoa ...,7.096017,0.823416,100,1.5,0.75
64,a60d2aa5-2019-04-18T17:14:53Z-00001-000,Russell Hantz Should Have Won Survivor: Samoa ...,4.098500,0.807318,100,1.5,0.75
2291,a8be11b0-2019-04-18T18:58:55Z-00000-000,Cannabis should be legalised Tiresome. I was h...,4.802697,0.805814,100,1.5,0.75
149,197ac9b-2019-04-18T19:47:57Z-00002-000,Baseball is a better sport than basketball. Al...,4.237482,0.803353,100,1.5,0.75


Serial Hyprid Results for dataset 2


,doc_id,text,bm25_candidate_score,word2vec_rerank_score,candidate_k,k1,b
2478,2559,What are the best ways to lose weight?,18.560818,1.000000,100,1.5,0.75
2624,2712,What are the best way of loose the weight?,12.566060,0.999821,100,1.5,0.75
2623,2711,What is the best method of losing weight?,14.414871,0.999735,100,1.5,0.75
1853,1919,How could I gain weight in a healthy way?,9.006859,0.999579,100,1.5,0.75
2360,2441,What is the best way to say 1?,7.568052,0.999565,100,1.5,0.75


In [43]:
# مطابقة وترتيب النتائج باستخدام التمثيل الهجين المتوازي
# في هذه الطريقة نحسب درجتين لكل وثيقة
# درجة من نموذج الاحتمالية ودرجة من المتجهات الدلالية
# ثم ندمج الدرجتين ونرتب النتائج حسب الدرجة النهائية

parallel_matching_ranking_1 = parallel_hybrid_search(
    query=test_query,
    docs_df=work1_df,
    tokenized_corpus=tokenized_corpus_1,
    word2vec_model=word2vec_model_1,
    word2vec_matrix=word2vec_matrix_1,
    top_k=5,
    alpha=0.6,
    k1=1.5,
    b=0.75
)

parallel_matching_ranking_2 = parallel_hybrid_search(
    query=test_query,
    docs_df=work2_df,
    tokenized_corpus=tokenized_corpus_2,
    word2vec_model=word2vec_model_2,
    word2vec_matrix=word2vec_matrix_2,
    top_k=5,
    alpha=0.6,
    k1=1.5,
    b=0.75
)

print("Parallel Hybrid  Search Results for dataset 1")
display(parallel_matching_ranking_1)

print("Parallel Hybrid Search Results for dataset 2")
display(parallel_matching_ranking_2)

Parallel Hybrid  Search Results for dataset 1


,doc_id,text,bm25_score_norm,word2vec_score_norm,final_hybrid_score,alpha,k1,b
1336,a88e4c42-2019-04-18T19:40:07Z-00003-000,I (im_always_right) will lose this debate Oh I...,1.000000,0.999739,0.999896,0.6,1.5,0.75
1061,2c5282d9-2019-04-18T19:39:34Z-00001-000,Does he fit the definition of a terorist? Note...,0.940967,0.867580,0.911612,0.6,1.5,0.75
65,a60d2aa5-2019-04-18T17:14:53Z-00002-000,Russell Hantz Should Have Won Survivor: Samoa ...,0.847745,0.998682,0.908120,0.6,1.5,0.75
1047,8840debf-2019-04-18T13:05:37Z-00004-000,swimming is a better sport than soccer So my o...,0.880871,0.943985,0.906117,0.6,1.5,0.75
512,f2f66016-2019-04-18T16:48:35Z-00002-000,Pokemon is freakin creepy. You disregard the k...,0.931383,0.845520,0.897038,0.6,1.5,0.75


Parallel Hybrid Search Results for dataset 2


,doc_id,text,bm25_score_norm,word2vec_score_norm,final_hybrid_score,alpha,k1,b
2478,2559,What are the best ways to lose weight?,1.000000,1.000000,1.000000,0.6,1.5,0.75
2623,2711,What is the best method of losing weight?,0.776629,0.999752,0.865878,0.6,1.5,0.75
2477,2558,Why do I not lose weight when I throw up?,0.697576,0.999248,0.818245,0.6,1.5,0.75
2624,2712,What are the best way of loose the weight?,0.677021,0.999832,0.806145,0.6,1.5,0.75
1633,1693,How do I lose weight without doing any sport?,0.631108,0.999326,0.778395,0.6,1.5,0.75


In [1]:
# تحميل المكتبات والأدوات اللازمة بعد إعادة فتح النوتبوك
# هذه الخلية لا تعيد تدريب أو تحميل الداتاسيت
# فقط تجهز المكتبات والدوال الأساسية للعمل

import os
import re
import pickle
import nltk
import spacy
import pandas as pd
import numpy as np

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from gensim.models import Word2Vec
from rank_bm25 import BM25Okapi
from sklearn.metrics.pairwise import cosine_similarity

nltk.download("stopwords")

stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

try:
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
    print("تم تحميل أداة المعالجة اللغوية بنجاح")
except:
    print("لم يتم العثور على أداة المعالجة اللغوية")
    print("إذا احتجتها شغل خلية التنصيب الخاصة بها")

SAVE_DIR = "ir_project_saved_files"

print("تم تجهيز المكتبات والأدوات")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


تم تحميل أداة المعالجة اللغوية بنجاح
تم تجهيز المكتبات والأدوات


In [2]:
# تحميل كل الملفات المحفوظة حتى نهاية الطلب السادس
# هذه الخلية تسترجع البيانات والتمثيلات والفهارس والنتائج المحفوظة
# الهدف أن نكمل من حيث توقفنا بدون إعادة التحميل والمعالجة والتدريب

import os
import pickle
import pandas as pd
import numpy as np
from gensim.models import Word2Vec

SAVE_DIR = "ir_project_saved_files"


def load_pickle_if_exists(file_name):
    path = os.path.join(SAVE_DIR, file_name)
    
    if os.path.exists(path):
        with open(path, "rb") as f:
            return pickle.load(f)
    else:
        print("ملف غير موجود:", file_name)
        return None


def load_csv_if_exists(file_name):
    path = os.path.join(SAVE_DIR, file_name)
    
    if os.path.exists(path):
        return pd.read_csv(path)
    else:
        print("ملف غير موجود:", file_name)
        return None


def load_numpy_if_exists(file_name):
    path = os.path.join(SAVE_DIR, file_name)
    
    if os.path.exists(path):
        return np.load(path)
    else:
        print("ملف غير موجود:", file_name)
        return None


def load_word2vec_if_exists(file_name):
    path = os.path.join(SAVE_DIR, file_name)
    
    if os.path.exists(path):
        return Word2Vec.load(path)
    else:
        print("ملف غير موجود:", file_name)
        return None


print("=" * 100)
print("تحميل البيانات المعالجة")
print("=" * 100)

docs1_df = load_csv_if_exists("dataset1_processed_docs.csv")
docs2_df = load_csv_if_exists("dataset2_processed_docs.csv")

work1_df = load_csv_if_exists("work_dataset1.csv")
work2_df = load_csv_if_exists("work_dataset2.csv")


print("\n" + "=" * 100)
print("تحميل تمثيل التردد والوزن")
print("=" * 100)

tfidf_vectorizer_1 = load_pickle_if_exists("tfidf_vectorizer_1.pkl")
tfidf_vectorizer_2 = load_pickle_if_exists("tfidf_vectorizer_2.pkl")

tfidf_matrix_1 = load_pickle_if_exists("tfidf_matrix_1.pkl")
tfidf_matrix_2 = load_pickle_if_exists("tfidf_matrix_2.pkl")


print("\n" + "=" * 100)
print("تحميل تمثيل المتجهات الدلالية")
print("=" * 100)

word2vec_model_1 = load_word2vec_if_exists("word2vec_model_dataset1.model")
word2vec_model_2 = load_word2vec_if_exists("word2vec_model_dataset2.model")

word2vec_matrix_1 = load_numpy_if_exists("word2vec_matrix_1.npy")
word2vec_matrix_2 = load_numpy_if_exists("word2vec_matrix_2.npy")


print("\n" + "=" * 100)
print("تحميل بيانات النموذج الاحتمالي")
print("=" * 100)

tokenized_corpus_1 = load_pickle_if_exists("tokenized_corpus_1.pkl")
tokenized_corpus_2 = load_pickle_if_exists("tokenized_corpus_2.pkl")


print("\n" + "=" * 100)
print("تحميل الفهارس")
print("=" * 100)

inverted_index_1 = load_pickle_if_exists("inverted_index_1.pkl")
inverted_index_2 = load_pickle_if_exists("inverted_index_2.pkl")


print("\n" + "=" * 100)
print("تحميل نتائج الطلبات السابقة إن وجدت")
print("=" * 100)

tfidf_results_1 = load_csv_if_exists("tfidf_results_dataset1.csv")
tfidf_results_2 = load_csv_if_exists("tfidf_results_dataset2.csv")

word2vec_results_1 = load_csv_if_exists("word2vec_results_dataset1.csv")
word2vec_results_2 = load_csv_if_exists("word2vec_results_dataset2.csv")

bm25_results_1 = load_csv_if_exists("bm25_results_dataset1.csv")
bm25_results_2 = load_csv_if_exists("bm25_results_dataset2.csv")

serial_results_1 = load_csv_if_exists("serial_hybrid_results_dataset1.csv")
serial_results_2 = load_csv_if_exists("serial_hybrid_results_dataset2.csv")

parallel_results_1 = load_csv_if_exists("parallel_hybrid_results_dataset1.csv")
parallel_results_2 = load_csv_if_exists("parallel_hybrid_results_dataset2.csv")

index_results_1 = load_csv_if_exists("inverted_index_results_dataset1.csv")
index_results_2 = load_csv_if_exists("inverted_index_results_dataset2.csv")

query_processing_summary_df = load_csv_if_exists("query_processing_summary_english.csv")

query_refinement_summary = load_csv_if_exists("query_refinement_summary.csv")

matching_ranking_summary = load_csv_if_exists("matching_ranking_summary.csv")


print("\n" + "=" * 100)
print("ملخص الملفات المحملة")
print("=" * 100)

if work1_df is not None:
    print("حجم جدول العمل الأول:", work1_df.shape)

if work2_df is not None:
    print("حجم جدول العمل الثاني:", work2_df.shape)

if tfidf_matrix_1 is not None:
    print("حجم مصفوفة التردد والوزن الأولى:", tfidf_matrix_1.shape)

if tfidf_matrix_2 is not None:
    print("حجم مصفوفة التردد والوزن الثانية:", tfidf_matrix_2.shape)

if word2vec_matrix_1 is not None:
    print("حجم مصفوفة المتجهات الدلالية الأولى:", word2vec_matrix_1.shape)

if word2vec_matrix_2 is not None:
    print("حجم مصفوفة المتجهات الدلالية الثانية:", word2vec_matrix_2.shape)

if tokenized_corpus_1 is not None:
    print("عدد وثائق النموذج الاحتمالي الأولى:", len(tokenized_corpus_1))

if tokenized_corpus_2 is not None:
    print("عدد وثائق النموذج الاحتمالي الثانية:", len(tokenized_corpus_2))

if inverted_index_1 is not None:
    print("عدد مصطلحات الفهرس الأول:", len(inverted_index_1))

if inverted_index_2 is not None:
    print("عدد مصطلحات الفهرس الثاني:", len(inverted_index_2))

print("\nتم تحميل الملفات الأساسية")

تحميل البيانات المعالجة

تحميل تمثيل التردد والوزن

تحميل تمثيل المتجهات الدلالية

تحميل بيانات النموذج الاحتمالي

تحميل الفهارس

تحميل نتائج الطلبات السابقة إن وجدت

ملخص الملفات المحملة
حجم جدول العمل الأول: (3000, 3)
حجم جدول العمل الثاني: (3000, 3)
حجم مصفوفة التردد والوزن الأولى: (3000, 17463)
حجم مصفوفة التردد والوزن الثانية: (3000, 3772)
حجم مصفوفة المتجهات الدلالية الأولى: (3000, 100)
حجم مصفوفة المتجهات الدلالية الثانية: (3000, 100)
عدد وثائق النموذج الاحتمالي الأولى: 3000
عدد وثائق النموذج الاحتمالي الثانية: 3000
عدد مصطلحات الفهرس الأول: 17465
عدد مصطلحات الفهرس الثاني: 3772

تم تحميل الملفات الأساسية


In [3]:
# إعادة تعريف الدوال المطلوبة للعمل بعد تحميل الملفات
# هذه الخلية تعيد تعريف دوال المعالجة والبحث والتحسين والمطابقة والترتيب
# لا تعيد هذه الخلية تحميل الداتاسيت ولا تدريب النماذج

def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def preprocess_stemming(text):
    text = normalize_text(text)
    words = text.split()
    
    tokens = []
    
    for word in words:
        if word not in stop_words and len(word) > 2:
            tokens.append(stemmer.stem(word))
    
    return " ".join(tokens)


def document_to_vector(tokens, model, vector_size=100):
    vectors = []
    
    for token in tokens:
        if token in model.wv:
            vectors.append(model.wv[token])
    
    if len(vectors) == 0:
        return np.zeros(vector_size)
    
    return np.mean(vectors, axis=0)


def min_max_normalize(scores):
    scores = np.array(scores)
    
    min_score = scores.min()
    max_score = scores.max()
    
    if max_score - min_score == 0:
        return np.zeros_like(scores)
    
    return (scores - min_score) / (max_score - min_score)


def search_tfidf(query, docs_df, vectorizer, tfidf_matrix, top_k=10):
    processed_query = preprocess_stemming(query)
    query_vector = vectorizer.transform([processed_query])
    
    scores = cosine_similarity(query_vector, tfidf_matrix).flatten()
    top_indices = scores.argsort()[::-1][:top_k]
    
    results = docs_df.iloc[top_indices][["doc_id", "text"]].copy()
    results["tfidf_score"] = scores[top_indices]
    
    return results


def search_word2vec(query, docs_df, word2vec_model, word2vec_matrix, top_k=10):
    processed_query = preprocess_stemming(query)
    query_tokens = processed_query.split()
    
    query_vector = document_to_vector(
        query_tokens,
        word2vec_model,
        vector_size=word2vec_matrix.shape[1]
    ).reshape(1, -1)
    
    scores = cosine_similarity(query_vector, word2vec_matrix).flatten()
    top_indices = scores.argsort()[::-1][:top_k]
    
    results = docs_df.iloc[top_indices][["doc_id", "text"]].copy()
    results["word2vec_score"] = scores[top_indices]
    
    return results


def search_bm25(query, docs_df, tokenized_corpus, top_k=10, k1=1.5, b=0.75):
    processed_query = preprocess_stemming(query)
    tokenized_query = processed_query.split()
    
    bm25_model = BM25Okapi(
        tokenized_corpus,
        k1=k1,
        b=b
    )
    
    scores = np.array(
        bm25_model.get_scores(tokenized_query)
    )
    
    top_indices = scores.argsort()[::-1][:top_k]
    
    results = docs_df.iloc[top_indices][["doc_id", "text"]].copy()
    results["bm25_score"] = scores[top_indices]
    results["k1"] = k1
    results["b"] = b
    
    return results


def match_and_rank_inverted_index(query, docs_df, inverted_index, top_k=10):
    processed_query = preprocess_stemming(query)
    query_tokens = processed_query.split()
    
    document_scores = {}
    
    for token in query_tokens:
        if token in inverted_index:
            posting_list = inverted_index[token]
            
            for doc_id, frequency in posting_list.items():
                doc_id = str(doc_id)
                
                if doc_id not in document_scores:
                    document_scores[doc_id] = 0
                
                document_scores[doc_id] += frequency
    
    if len(document_scores) == 0:
        return pd.DataFrame([
            {
                "rank": "-",
                "doc_id": "-",
                "matching_score": 0,
                "text": "لا توجد وثائق مطابقة لكلمات الاستعلام داخل الفهرس الحالي"
            }
        ])
    
    temp_docs_df = docs_df.copy()
    temp_docs_df["doc_id"] = temp_docs_df["doc_id"].astype(str)
    
    results = temp_docs_df[
        temp_docs_df["doc_id"].isin(document_scores.keys())
    ][["doc_id", "text"]].copy()
    
    results["matching_score"] = results["doc_id"].map(document_scores)
    
    results = results.sort_values(
        by="matching_score",
        ascending=False
    ).head(top_k).copy()
    
    results["rank"] = range(1, len(results) + 1)
    
    results = results[["rank", "doc_id", "matching_score", "text"]]
    
    return results


def serial_hybrid_search(
    query,
    docs_df,
    tokenized_corpus,
    word2vec_model,
    word2vec_matrix,
    top_k=10,
    candidate_k=100,
    k1=1.5,
    b=0.75
):
    processed_query = preprocess_stemming(query)
    tokenized_query = processed_query.split()
    
    bm25_model = BM25Okapi(
        tokenized_corpus,
        k1=k1,
        b=b
    )
    
    bm25_scores = np.array(
        bm25_model.get_scores(tokenized_query)
    )
    
    candidate_k = min(candidate_k, len(docs_df))
    candidate_indices = bm25_scores.argsort()[::-1][:candidate_k]
    
    query_vector = document_to_vector(
        tokenized_query,
        word2vec_model,
        vector_size=word2vec_matrix.shape[1]
    ).reshape(1, -1)
    
    candidate_embeddings = word2vec_matrix[candidate_indices]
    
    embedding_scores = cosine_similarity(
        query_vector,
        candidate_embeddings
    ).flatten()
    
    reranked_positions = embedding_scores.argsort()[::-1][:top_k]
    final_indices = candidate_indices[reranked_positions]
    
    results = docs_df.iloc[final_indices][["doc_id", "text"]].copy()
    results["bm25_candidate_score"] = bm25_scores[final_indices]
    results["word2vec_rerank_score"] = embedding_scores[reranked_positions]
    results["candidate_k"] = candidate_k
    results["k1"] = k1
    results["b"] = b
    
    return results


def parallel_hybrid_search(
    query,
    docs_df,
    tokenized_corpus,
    word2vec_model,
    word2vec_matrix,
    top_k=10,
    alpha=0.6,
    k1=1.5,
    b=0.75
):
    processed_query = preprocess_stemming(query)
    tokenized_query = processed_query.split()
    
    bm25_model = BM25Okapi(
        tokenized_corpus,
        k1=k1,
        b=b
    )
    
    bm25_scores = np.array(
        bm25_model.get_scores(tokenized_query)
    )
    
    query_vector = document_to_vector(
        tokenized_query,
        word2vec_model,
        vector_size=word2vec_matrix.shape[1]
    ).reshape(1, -1)
    
    word2vec_scores = cosine_similarity(
        query_vector,
        word2vec_matrix
    ).flatten()
    
    bm25_norm = min_max_normalize(bm25_scores)
    word2vec_norm = min_max_normalize(word2vec_scores)
    
    final_scores = alpha * bm25_norm + (1 - alpha) * word2vec_norm
    
    top_indices = final_scores.argsort()[::-1][:top_k]
    
    results = docs_df.iloc[top_indices][["doc_id", "text"]].copy()
    results["bm25_score_norm"] = bm25_norm[top_indices]
    results["word2vec_score_norm"] = word2vec_norm[top_indices]
    results["final_hybrid_score"] = final_scores[top_indices]
    results["alpha"] = alpha
    results["k1"] = k1
    results["b"] = b
    
    return results


def process_query(query):
    processed_query = preprocess_stemming(query)
    query_tokens = processed_query.split()
    
    return {
        "original_query": query,
        "processed_query": processed_query,
        "query_tokens": query_tokens
    }


def represent_query_tfidf(query, tfidf_vectorizer):
    query_info = process_query(query)
    query_vector = tfidf_vectorizer.transform([query_info["processed_query"]])
    return query_vector


def represent_query_word2vec(query, word2vec_model, vector_size=100):
    query_info = process_query(query)
    
    query_vector = document_to_vector(
        query_info["query_tokens"],
        word2vec_model,
        vector_size=vector_size
    )
    
    return query_vector


def represent_query_bm25(query):
    query_info = process_query(query)
    return query_info["query_tokens"]


def query_terms_in_index(query, inverted_index):
    query_info = process_query(query)
    
    terms_status = []
    
    for token in query_info["query_tokens"]:
        if token in inverted_index:
            document_frequency = len(inverted_index[token])
        else:
            document_frequency = 0
        
        terms_status.append({
            "query_term": token,
            "exists_in_index": token in inverted_index,
            "document_frequency": document_frequency
        })
    
    return pd.DataFrame(terms_status)


def complete_query_representation(query, dataset_number=1):
    query_info = process_query(query)
    
    if dataset_number == 1:
        tfidf_vectorizer = tfidf_vectorizer_1
        word2vec_model = word2vec_model_1
        word2vec_matrix = word2vec_matrix_1
        inverted_index = inverted_index_1
    else:
        tfidf_vectorizer = tfidf_vectorizer_2
        word2vec_model = word2vec_model_2
        word2vec_matrix = word2vec_matrix_2
        inverted_index = inverted_index_2
    
    tfidf_vector = represent_query_tfidf(
        query=query,
        tfidf_vectorizer=tfidf_vectorizer
    )
    
    word2vec_vector = represent_query_word2vec(
        query=query,
        word2vec_model=word2vec_model,
        vector_size=word2vec_matrix.shape[1]
    )
    
    bm25_tokens = represent_query_bm25(query)
    
    index_status = query_terms_in_index(
        query=query,
        inverted_index=inverted_index
    )
    
    return {
        "query_info": query_info,
        "tfidf_vector": tfidf_vector,
        "word2vec_vector": word2vec_vector,
        "bm25_tokens": bm25_tokens,
        "index_status": index_status
    }


spelling_corrections = {
    "weigth": "weight",
    "wieght": "weight",
    "wheight": "weight",
    "loose": "lose",
    "losingg": "losing",
    "helth": "health",
    "excersise": "exercise",
    "programing": "programming",
    "lern": "learn",
    "lerning": "learning",
    "tecnology": "technology",
    "enviroment": "environment"
}


synonym_dictionary = {
    "weight": ["loss", "diet", "fitness"],
    "lose": ["weight", "loss", "diet"],
    "health": ["medical", "fitness", "wellness"],
    "exercise": ["workout", "fitness", "training"],
    "programming": ["coding", "software", "development"],
    "learn": ["study", "education", "training"],
    "car": ["automobile", "vehicle"],
    "climate": ["environment", "weather"],
    "movie": ["film", "cinema"],
    "food": ["meal", "diet", "nutrition"]
}


user_search_history = [
    "weight loss diet",
    "best exercise for fitness",
    "healthy food and nutrition",
    "learn programming basics"
]


def correct_query_spelling(query):
    query_info = process_query(query)
    tokens = query_info["query_tokens"]
    
    corrected_tokens = []
    
    for token in tokens:
        if token in spelling_corrections:
            corrected_tokens.append(spelling_corrections[token])
        else:
            corrected_tokens.append(token)
    
    corrected_query = " ".join(corrected_tokens)
    
    return corrected_query


def expand_query_with_synonyms(query, max_synonyms_per_term=2):
    query_info = process_query(query)
    tokens = query_info["query_tokens"]
    
    expanded_tokens = []
    
    for token in tokens:
        expanded_tokens.append(token)
        
        if token in synonym_dictionary:
            related_terms = synonym_dictionary[token][:max_synonyms_per_term]
            
            for related_term in related_terms:
                if related_term not in expanded_tokens:
                    expanded_tokens.append(related_term)
    
    expanded_query = " ".join(expanded_tokens)
    
    return expanded_query


def suggest_query_from_history(query, search_history, top_k=3):
    processed_query = preprocess_stemming(query)
    query_tokens = set(processed_query.split())
    
    suggestions = []
    
    for old_query in search_history:
        processed_old_query = preprocess_stemming(old_query)
        old_query_tokens = set(processed_old_query.split())
        
        common_terms = query_tokens.intersection(old_query_tokens)
        score = len(common_terms)
        
        if score > 0:
            suggestions.append({
                "suggested_query": old_query,
                "similarity_score": score
            })
    
    suggestions = sorted(
        suggestions,
        key=lambda x: x["similarity_score"],
        reverse=True
    )
    
    return pd.DataFrame(suggestions[:top_k])


def refine_query(query, use_spelling=True, use_expansion=True, use_history=True):
    original_query = query
    processed_original = process_query(original_query)["processed_query"]
    
    if use_spelling:
        corrected_query = correct_query_spelling(original_query)
    else:
        corrected_query = processed_original
    
    if use_expansion:
        expanded_query = expand_query_with_synonyms(corrected_query)
    else:
        expanded_query = corrected_query
    
    if use_history:
        history_suggestions = suggest_query_from_history(
            query=expanded_query,
            search_history=user_search_history,
            top_k=3
        )
    else:
        history_suggestions = pd.DataFrame()
    
    refined_query = expanded_query
    
    refinement_report = {
        "original_query": original_query,
        "processed_original_query": processed_original,
        "corrected_query": corrected_query,
        "expanded_query": expanded_query,
        "refined_query": refined_query,
        "history_suggestions": history_suggestions
    }
    
    return refinement_report


def rank_results_by_score(results_df, score_column, top_k=10):
    ranked_results = results_df.sort_values(
        by=score_column,
        ascending=False
    ).head(top_k).copy()
    
    ranked_results["rank"] = range(1, len(ranked_results) + 1)
    
    columns_order = ["rank", "doc_id", score_column, "text"]
    
    available_columns = [
        column
        for column in columns_order
        if column in ranked_results.columns
    ]
    
    return ranked_results[available_columns]


def match_and_rank_tfidf(query, docs_df, vectorizer, tfidf_matrix, top_k=10):
    processed_query = preprocess_stemming(query)
    query_vector = vectorizer.transform([processed_query])
    
    similarity_scores = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()
    
    results = docs_df[["doc_id", "text"]].copy()
    results["similarity_score"] = similarity_scores
    
    ranked_results = rank_results_by_score(
        results_df=results,
        score_column="similarity_score",
        top_k=top_k
    )
    
    return ranked_results


def match_and_rank_word2vec(query, docs_df, word2vec_model, word2vec_matrix, top_k=10):
    processed_query = preprocess_stemming(query)
    query_tokens = processed_query.split()
    
    query_vector = document_to_vector(
        query_tokens,
        word2vec_model,
        vector_size=word2vec_matrix.shape[1]
    ).reshape(1, -1)
    
    similarity_scores = cosine_similarity(
        query_vector,
        word2vec_matrix
    ).flatten()
    
    results = docs_df[["doc_id", "text"]].copy()
    results["similarity_score"] = similarity_scores
    
    ranked_results = rank_results_by_score(
        results_df=results,
        score_column="similarity_score",
        top_k=top_k
    )
    
    return ranked_results


def match_and_rank_bm25(query, docs_df, tokenized_corpus, top_k=10, k1=1.5, b=0.75):
    results = search_bm25(
        query=query,
        docs_df=docs_df,
        tokenized_corpus=tokenized_corpus,
        top_k=len(docs_df),
        k1=k1,
        b=b
    )
    
    ranked_results = rank_results_by_score(
        results_df=results,
        score_column="bm25_score",
        top_k=top_k
    )
    
    return ranked_results


print("تمت إعادة تعريف كل الدوال المطلوبة بنجاح")

تمت إعادة تعريف كل الدوال المطلوبة بنجاح


In [4]:
# اختبار شامل بعد تحميل الملفات وإعادة تعريف الدوال
# هذه الخلية تتأكد أن كل أجزاء المشروع من الطلب الأول إلى السادس تعمل بعد إعادة فتح النوتبوك

test_query = "what is the best way to lose weight"
test_refinement_query = "weigth loss"

print("=" * 100)
print("اختبار شامل بعد استرجاع العمل المحفوظ")
print("=" * 100)


print("\n" + "=" * 100)
print("الطلب الأول: البيانات والمعالجة المسبقة")
print("=" * 100)

print("حجم بيانات العمل الأولى:", work1_df.shape)
print("حجم بيانات العمل الثانية:", work2_df.shape)

print("\nمثال من النص الأصلي والمعالج:")
display(work1_df[["doc_id", "text", "processed_text"]].head(3))


print("\n" + "=" * 100)
print("الطلب الثاني: تمثيل الوثائق")
print("=" * 100)

print("حجم مصفوفة التردد والوزن الأولى:", tfidf_matrix_1.shape)
print("حجم مصفوفة التردد والوزن الثانية:", tfidf_matrix_2.shape)

print("حجم مصفوفة المتجهات الدلالية الأولى:", word2vec_matrix_1.shape)
print("حجم مصفوفة المتجهات الدلالية الثانية:", word2vec_matrix_2.shape)

print("عدد وثائق النموذج الاحتمالي الأولى:", len(tokenized_corpus_1))
print("عدد وثائق النموذج الاحتمالي الثانية:", len(tokenized_corpus_2))


print("\nنتائج طريقة التردد والوزن:")
display(search_tfidf(
    query=test_query,
    docs_df=work1_df,
    vectorizer=tfidf_vectorizer_1,
    tfidf_matrix=tfidf_matrix_1,
    top_k=3
))


print("\nنتائج المتجهات الدلالية:")
display(search_word2vec(
    query=test_query,
    docs_df=work1_df,
    word2vec_model=word2vec_model_1,
    word2vec_matrix=word2vec_matrix_1,
    top_k=3
))


print("\nنتائج النموذج الاحتمالي:")
display(search_bm25(
    query=test_query,
    docs_df=work1_df,
    tokenized_corpus=tokenized_corpus_1,
    top_k=3,
    k1=1.5,
    b=0.75
))


print("\nنتائج التمثيل الهجين التسلسلي:")
display(serial_hybrid_search(
    query=test_query,
    docs_df=work1_df,
    tokenized_corpus=tokenized_corpus_1,
    word2vec_model=word2vec_model_1,
    word2vec_matrix=word2vec_matrix_1,
    top_k=3,
    candidate_k=100,
    k1=1.5,
    b=0.75
))


print("\nنتائج التمثيل الهجين المتوازي:")
display(parallel_hybrid_search(
    query=test_query,
    docs_df=work1_df,
    tokenized_corpus=tokenized_corpus_1,
    word2vec_model=word2vec_model_1,
    word2vec_matrix=word2vec_matrix_1,
    top_k=3,
    alpha=0.6,
    k1=1.5,
    b=0.75
))


print("\n" + "=" * 100)
print("الطلب الثالث: الفهرسة")
print("=" * 100)

print("عدد مصطلحات الفهرس الأول:", len(inverted_index_1))
print("عدد مصطلحات الفهرس الثاني:", len(inverted_index_2))

print("\nنتائج الفهرس المعكوس للداتاسيت الأولى:")
display(match_and_rank_inverted_index(
    query=test_query,
    docs_df=work1_df,
    inverted_index=inverted_index_1,
    top_k=3
))

print("\nنتائج الفهرس المعكوس للداتاسيت الثانية:")
display(match_and_rank_inverted_index(
    query=test_query,
    docs_df=work2_df,
    inverted_index=inverted_index_2,
    top_k=3
))


print("\n" + "=" * 100)
print("الطلب الرابع: معالجة الاستعلام")
print("=" * 100)

query_representation_1 = complete_query_representation(
    query=test_query,
    dataset_number=1
)

print("الاستعلام الأصلي:", query_representation_1["query_info"]["original_query"])
print("الاستعلام بعد المعالجة:", query_representation_1["query_info"]["processed_query"])
print("كلمات الاستعلام:", query_representation_1["query_info"]["query_tokens"])
print("حجم تمثيل التردد والوزن:", query_representation_1["tfidf_vector"].shape)
print("حجم تمثيل المتجهات الدلالية:", query_representation_1["word2vec_vector"].shape)
print("كلمات النموذج الاحتمالي:", query_representation_1["bm25_tokens"])

print("\nحالة كلمات الاستعلام داخل الفهرس:")
display(query_representation_1["index_status"])


print("\n" + "=" * 100)
print("الطلب الخامس: تحسين الاستعلام")
print("=" * 100)

refinement_report = refine_query(
    query=test_refinement_query,
    use_spelling=True,
    use_expansion=True,
    use_history=True
)

print("الاستعلام الأصلي:", refinement_report["original_query"])
print("الاستعلام بعد المعالجة:", refinement_report["processed_original_query"])
print("الاستعلام بعد التصحيح:", refinement_report["corrected_query"])
print("الاستعلام بعد التوسيع:", refinement_report["expanded_query"])
print("الاستعلام النهائي المحسن:", refinement_report["refined_query"])

print("\nاقتراحات من سجل البحث:")
display(refinement_report["history_suggestions"])

print("\nنتائج قبل تحسين الاستعلام:")
display(search_bm25(
    query=refinement_report["original_query"],
    docs_df=work1_df,
    tokenized_corpus=tokenized_corpus_1,
    top_k=3,
    k1=1.5,
    b=0.75
))

print("\nنتائج بعد تحسين الاستعلام:")
display(search_bm25(
    query=refinement_report["refined_query"],
    docs_df=work1_df,
    tokenized_corpus=tokenized_corpus_1,
    top_k=3,
    k1=1.5,
    b=0.75
))


print("\n" + "=" * 100)
print("الطلب السادس: مطابقة الاستعلام وترتيب النتائج")
print("=" * 100)

print("\nمطابقة وترتيب باستخدام طريقة التردد والوزن:")
display(match_and_rank_tfidf(
    query=test_query,
    docs_df=work1_df,
    vectorizer=tfidf_vectorizer_1,
    tfidf_matrix=tfidf_matrix_1,
    top_k=3
))

print("\nمطابقة وترتيب باستخدام المتجهات الدلالية:")
display(match_and_rank_word2vec(
    query=test_query,
    docs_df=work1_df,
    word2vec_model=word2vec_model_1,
    word2vec_matrix=word2vec_matrix_1,
    top_k=3
))

print("\nمطابقة وترتيب باستخدام النموذج الاحتمالي:")
display(match_and_rank_bm25(
    query=test_query,
    docs_df=work1_df,
    tokenized_corpus=tokenized_corpus_1,
    top_k=3,
    k1=1.5,
    b=0.75
))

print("\nمطابقة وترتيب باستخدام الفهرس المعكوس:")
display(match_and_rank_inverted_index(
    query=test_query,
    docs_df=work1_df,
    inverted_index=inverted_index_1,
    top_k=3
))


print("\n" + "=" * 100)
print("انتهى الاختبار الشامل بنجاح إذا ظهرت الجداول السابقة بدون أخطاء")
print("=" * 100)

اختبار شامل بعد استرجاع العمل المحفوظ

الطلب الأول: البيانات والمعالجة المسبقة
حجم بيانات العمل الأولى: (3000, 3)
حجم بيانات العمل الثانية: (3000, 3)

مثال من النص الأصلي والمعالج:


,doc_id,text,processed_text
0,c67482ba-2019-04-18T13:32:05Z-00000-000,Contraceptive Forms for High School Students M...,contracept form high school student oppon forf...
1,c67482ba-2019-04-18T13:32:05Z-00001-000,Contraceptive Forms for High School Students H...,contracept form high school student propos sch...
2,c67482ba-2019-04-18T13:32:05Z-00002-000,Contraceptive Forms for High School Students S...,contracept form high school student school com...



الطلب الثاني: تمثيل الوثائق
حجم مصفوفة التردد والوزن الأولى: (3000, 17463)
حجم مصفوفة التردد والوزن الثانية: (3000, 3772)
حجم مصفوفة المتجهات الدلالية الأولى: (3000, 100)
حجم مصفوفة المتجهات الدلالية الثانية: (3000, 100)
عدد وثائق النموذج الاحتمالي الأولى: 3000
عدد وثائق النموذج الاحتمالي الثانية: 3000

نتائج طريقة التردد والوزن:


,doc_id,text,tfidf_score
1338,a88e4c42-2019-04-18T19:40:07Z-00005-000,I (im_always_right) will lose this debate if I...,0.374399
1336,a88e4c42-2019-04-18T19:40:07Z-00003-000,I (im_always_right) will lose this debate Oh I...,0.359070
1337,a88e4c42-2019-04-18T19:40:07Z-00004-000,I (im_always_right) will lose this debate I ne...,0.283284



نتائج المتجهات الدلالية:


...


مطابقة وترتيب باستخدام النموذج الاحتمالي:


,rank,doc_id,bm25_score,text
1336,1,a88e4c42-2019-04-18T19:40:07Z-00003-000,8.370463,I (im_always_right) will lose this debate Oh I...
1061,2,2c5282d9-2019-04-18T19:39:34Z-00001-000,7.876333,Does he fit the definition of a terorist? Note...
512,3,f2f66016-2019-04-18T16:48:35Z-00002-000,7.796105,Pokemon is freakin creepy. You disregard the k...



مطابقة وترتيب باستخدام الفهرس المعكوس:


,rank,doc_id,matching_score,text
1337,1,a88e4c42-2019-04-18T19:40:07Z-00004-000,12,I (im_always_right) will lose this debate I ne...
612,2,ef778452-2019-04-18T15:25:55Z-00006-000,11,Evolution vs creationism I will support creati...
125,3,a617b5f8-2019-04-18T12:37:07Z-00002-000,11,The US Constitution Should Be a Living Documen...



انتهى الاختبار الشامل بنجاح إذا ظهرت الجداول السابقة بدون أخطاء
